# Bengaluru Smart Mobility Intelligence
**An end-to-end AI/ML capstone on real Bengaluru traffic, weather, road-safety and enforcement data.**

This notebook is **standalone**: run it top to bottom in Google Colab (Runtime → Run all). It will ask you to upload the four CSV files if it cannot find them.

**Expected runtime:** about 20–30 minutes on a standard CPU runtime (most of it is hyper-parameter tuning and SHAP). Phase 18 trains the YOLO detector: on a **T4 GPU** runtime (Runtime → Change runtime type) it takes roughly 20–40 minutes; on a CPU runtime the notebook automatically uses a reduced 'lite' configuration (fewer images and epochs, lower accuracy) so the whole project still completes. Lower `N_ITER` in the setup cell for a quicker pass.

| Phase | What happens |
|---|---|
| 1 | Problem definition and the role of every dataset |
| 2–3 | Data collection, understanding and cleaning |
| 4 | Target verification, leakage audit, leak-free modelling table |
| 5–7 | Target distribution, chronological split, feature relationships, multicollinearity |
| 8 | Feature engineering and feature selection |
| 9–12 | Model comparison, hyper-parameter tuning, cross-validation, final model selection, deep-learning (GRU/LSTM) comparison |
| 13–15 | Final evaluation, residual and error analysis, explainable AI (permutation + SHAP), saved models and prediction system |
| 16–17 | Mobility intelligence layer, incident-risk model, road safety, enforcement, fused intelligence report |
| **18** | **YOLO CCTV vehicle detection (BMD-45): detector, PCU load, visual congestion index** |
| 19 | Dashboard |
| **20** | **Streamlit deployment — prediction, dashboards and the live CCTV detection tab** |
| **21** | **Documentation, presentation, resume text, final package and scoreboard** |

## Phase 1 — Problem definition

Two prediction tasks come out of the data. They answer different questions and are evaluated separately.

| Task | Target | Question | Inputs available at prediction time |
|---|---|---|---|
| **A. Forecast** | congestion level **tomorrow** on a road | "Where will traffic be bad tomorrow?" | today's readings, the last 7 days, tomorrow's weather forecast and calendar, the road |
| **B. What-if estimator** | congestion level **now** | "What congestion should we expect under these conditions?" | traffic volume, incidents, roadwork, public-transport usage, weather, calendar, the road — *not* the congestion-derived sensor readings |

**Dataset roles**

| Dataset | Role |
|---|---|
| Traffic dataset (8,936 road-days, 16 roads) | **Prediction dataset** for A and B, plus the main source of mobility insight and the dashboard |
| Weather (hourly, 2022 →) | **Feature source** (same-day weather and next-day forecast) |
| Road crashes 2007–2025 | **Supporting analytical dataset**: road-safety insight, severity factor for the risk index |
| BTP cases and fines | **Business-insight / dashboard dataset**: enforcement load and automation trend |
| BMD-45 images | **Vision module** (optional, Phase 19): vehicle detection and a visual congestion index |

**Leakage rules used throughout:** (1) a feature may only use information available *before* the prediction moment; (2) the split is **chronological**; (3) rows whose "today" is an interpolated filler are never used; (4) anything that suspiciously inflates R² is investigated, not celebrated.

## Phase 2 — Setup and data collection

In [ ]:
!pip install -q catboost shap python-pptx streamlit

In [ ]:
import os, re, json, glob, math, copy, random, shutil, subprocess, sys, warnings, textwrap, datetime as dt
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import plotly.express as px, plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
from sklearn.base import BaseEstimator, RegressorMixin, clone
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge, LogisticRegression, LinearRegression
from sklearn.ensemble import (RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor,
                              HistGradientBoostingRegressor, HistGradientBoostingClassifier)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit, KFold, RandomizedSearchCV, cross_validate, cross_val_score
from sklearn.feature_selection import mutual_info_regression
from sklearn.inspection import permutation_importance
from sklearn.metrics import (r2_score, mean_absolute_error, mean_squared_error, roc_auc_score, accuracy_score,
                             precision_score, recall_score, f1_score, brier_score_loss, mean_poisson_deviance)
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor, LGBMClassifier
from catboost import CatBoostRegressor

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED); np.random.seed(SEED)
N_ITER = int(os.environ.get("MOBILITY_N_ITER", 8))     # random-search iterations per model (raise for a final run)
ROOT = os.environ.get("PROJECT_ROOT", "/content")
PROJECT_DIR = f"{ROOT}/bengaluru_smart_mobility"
for sub in ["data", "models", "docs", "figures", "notebooks"]:
    os.makedirs(f"{PROJECT_DIR}/{sub}", exist_ok=True)
os.chdir(PROJECT_DIR)
sns.set_theme(style="whitegrid")
plt.rcParams.update({"figure.dpi": 110, "axes.titlesize": 12, "axes.titleweight": "bold"})

def savefig(name):
    plt.savefig(f"{PROJECT_DIR}/figures/{name}.png", bbox_inches="tight", dpi=140)

def slug(s):
    return re.sub(r"[^0-9a-zA-Z]+", "_", str(s)).strip("_").lower()

print("Project folder:", PROJECT_DIR)

In [ ]:
# ---- locate the four CSV files (uploads in /content, or the project's data/ folder) ----
NEEDED = {"traffic": ("traffic", "dataset"), "weather": ("weather",), "btp": ("btp",), "crash": ("crash",)}

def find_csv(*keys):
    for base in [ROOT, f"{PROJECT_DIR}/data"]:
        for dp, _, fn in os.walk(base):
            if dp.count(os.sep) - base.count(os.sep) > 2 or "sample_data" in dp:
                continue
            for f in fn:
                low = f.lower().replace(" ", "_").replace("-", "_")
                if low.endswith(".csv") and all(k in low for k in keys):
                    return os.path.join(dp, f)
    return None

paths = {k: find_csv(*v) for k, v in NEEDED.items()}
if any(p is None for p in paths.values()):
    print("Missing:", [k for k, p in paths.items() if p is None], "-> please upload the CSV files")
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        raise FileNotFoundError("Put the four CSV files in " + ROOT)
    paths = {k: find_csv(*v) for k, v in NEEDED.items()}
assert all(paths.values()), paths
for k, p in paths.items():
    print(f"{k:8s} <- {p}")

traffic = pd.read_csv(paths["traffic"])
traffic["Date"] = pd.to_datetime(traffic["Date"]).dt.normalize()
_lines = open(paths["weather"], encoding="utf-8-sig").read().splitlines()
weather = pd.read_csv(paths["weather"], skiprows=next(i for i, l in enumerate(_lines) if l.lower().startswith("time,")))
weather["time"] = pd.to_datetime(weather["time"])
btp = pd.read_csv(paths["btp"], encoding="utf-8-sig")
crash = pd.read_csv(paths["crash"], thousands=",", encoding="utf-8-sig")
for k, p in paths.items():
    if not os.path.exists(f"{PROJECT_DIR}/data/{os.path.basename(p)}"):
        shutil.copy(p, f"{PROJECT_DIR}/data/")

## Phase 3 — Data understanding and cleaning
Compact quality summary (the detailed EDA was done earlier and is not repeated). Two data-quality findings matter for modelling:
`Environmental Impact` is a rescaled copy of `Traffic Volume`, and the BTP table has a source inconsistency in 2022.

In [ ]:
def quality(name, df):
    return dict(dataset=name, rows=len(df), columns=df.shape[1], missing_cells=int(df.isna().sum().sum()),
                duplicate_rows=int(df.duplicated().sum()))
display(pd.DataFrame([quality("traffic", traffic), quality("weather (hourly)", weather),
                      quality("btp (yearly)", btp), quality("crash (yearly)", crash)]))

print("\nTraffic: %s -> %s | %d areas | %d roads" % (traffic["Date"].min().date(), traffic["Date"].max().date(),
      traffic["Area Name"].nunique(), traffic["Road/Intersection Name"].nunique()))
print("corr(Traffic Volume, Environmental Impact) = %.4f  -> duplicate information, Environmental Impact is dropped" %
      traffic["Traffic Volume"].corr(traffic["Environmental Impact"]))
print("weather: rain vs precipitation identical:", bool(np.allclose(weather["rain (mm)"], weather["precipitation (mm)"])))

case_cols = ["M.V.Act Cases", "K.P.Act Cases", "Towing Cases", "K.T.C Act Cases", "Automation cases"]
chk = (btp[case_cols].fillna(0).sum(axis=1) - btp["Total Cases Booked"])
bad = btp.loc[chk != 0, ["YEAR"]].assign(components_minus_total=chk[chk != 0])
print("BTP years where components != reported total:\n", bad.to_string(index=False) if len(bad) else "none")

## Phase 4 — Target verification and leakage audit  *(your Step 1)*

The earlier notebook defined the target as `Next_Day_Congestion` (the next *row's* congestion for the same road, kept when the **previous** gap was 1 day). Let's verify what that really is.

In [ ]:
KEYS_RAW = ["Area Name", "Road/Intersection Name"]
t0 = traffic.sort_values(KEYS_RAW + ["Date"]).reset_index(drop=True)
g0 = t0.groupby(KEYS_RAW)
t0["prev_gap"] = g0["Date"].diff().dt.days
t0["next_gap"] = g0["Date"].diff(-1).dt.days.abs()
t0["Next_Day_Congestion"] = g0["Congestion Level"].shift(-1)
old = t0[(t0["prev_gap"] == 1) & t0["Next_Day_Congestion"].notna()]
y_old = old["Next_Day_Congestion"]

print("Old target name:", y_old.name, "| rows:", len(old))
print(y_old.describe().round(2).to_string())
print("\nDays between a row and the row its target came from:")
print(old["next_gap"].value_counts().sort_index().head(6).to_string())
print("\nShare of 'next-day' targets that really are the next day: %.1f%%" % (100 * (old["next_gap"] == 1).mean()))
print("Share of targets sitting exactly at the ceiling of 100:     %.1f%%" % (100 * (y_old == 100).mean()))
true_nd = old[old["next_gap"] == 1]
print("Persistence R2 ('tomorrow = today'), truly-next-day rows:   %.3f  (corr %.3f)" %
      (r2_score(true_nd["Next_Day_Congestion"], true_nd["Congestion Level"]),
       true_nd["Next_Day_Congestion"].corr(true_nd["Congestion Level"])))

**What this tells us**
* The target is a continuous 0–100 congestion index, so regression is possible — but it is **right-censored**: about half of all values are exactly 100. A model cannot learn *how far above* 100 conditions were, and R² is capped by that.
* About **30 % of the old "next-day" targets are actually 2–17 days later**, so the old target mixes different horizons. We rebuild it so it is strictly *tomorrow*.
* Persistence (R² < 0) is **worse than predicting the average**: congestion on a road is largely independent from one day to the next. That is a property of the data and it limits any forecaster.

**Decision:** rebuild a daily road panel, define the target as the *real* congestion of the *next calendar day*, and only use rows where *today* is also a real observation.

In [ ]:
# ---- daily road panel: one row per road per calendar day -----------------------------------
OPS = ["traffic_volume", "average_speed", "travel_time_index", "congestion_level", "road_capacity_utilization",
       "incident_reports", "public_transport_usage", "traffic_signal_compliance", "parking_usage",
       "pedestrian_and_cyclist_count"]
WX = ["rain_sum", "temp_mean", "humidity_mean", "wind_max", "cloud_mean"]

tr = traffic.rename(columns={**{c: slug(c) for c in traffic.columns},
                             "Area Name": "area", "Road/Intersection Name": "road", "Date": "date"})
tr["roadwork"] = (tr["roadwork_and_construction_activity"].astype(str).str.lower() == "yes").astype(float)

w = weather.copy(); w["date"] = w["time"].dt.normalize()
weather_daily = w.groupby("date").agg(
    rain_sum=("rain (mm)", "sum"), temp_mean=("temperature_2m (°C)", "mean"),
    humidity_mean=("relative_humidity_2m (%)", "mean"), wind_max=("wind_speed_10m (km/h)", "max"),
    cloud_mean=("cloud_cover (%)", "mean")).reset_index()

def build_panel(tr, wd, max_gap=3):
    out = []
    for (a, r), g in tr.groupby(["area", "road"]):
        g = g.drop_duplicates("date").set_index("date").sort_index()
        g = g.reindex(pd.date_range(g.index.min(), g.index.max(), freq="D"))
        g["observed"] = g["traffic_volume"].notna().astype(int)
        g["cong_raw"] = g["congestion_level"]                      # untouched real values
        cols = OPS + ["roadwork"]
        # interpolation ONLY to build history windows (lags). Interpolated days are never used as 'today' or as a target.
        g[cols] = g[cols].interpolate(limit=max_gap, limit_area="inside")
        cg = g["congestion_level"]
        g["cong_lag1"] = cg.shift(1)
        g["cong_roll3"] = cg.rolling(3, min_periods=2).mean()
        g["cong_roll7"] = cg.rolling(7, min_periods=3).mean()
        g["cong_std7"] = cg.rolling(7, min_periods=3).std()
        g["sat_share7"] = (cg >= 99.9).astype(float).where(cg.notna()).rolling(7, min_periods=3).mean()
        g["vol_roll7"] = g["traffic_volume"].rolling(7, min_periods=3).mean()
        g["target"] = g["cong_raw"].shift(-1)                       # real congestion of the next calendar day, else NaN
        g["area"], g["road"] = a, r
        g.index.name = "date"
        out.append(g.reset_index())
    p = pd.concat(out, ignore_index=True).merge(wd, on="date", how="left")
    wn = wd.copy(); wn["date"] = wn["date"] - pd.Timedelta(days=1)       # tomorrow's weather, placed on today's row
    wn = wn.rename(columns={c: c + "_next" for c in WX})
    return p.merge(wn, on="date", how="left")

panel = build_panel(tr, weather_daily)
print("panel rows:", len(panel), "| real observations:", int(panel["observed"].sum()),
      "| interpolated filler days:", int((panel["observed"] == 0).sum()))

### Leakage demonstration (why "today must be a real observation")
If "today" is a filled-in day, interpolation draws on the *next* day — the very value we want to predict. The same model then looks far better than it really is. Same model, same chronological split, only the row filter differs:

In [ ]:
_f = OPS + ["roadwork", "rain_sum_next", "temp_mean_next", "cong_lag1", "cong_roll7", "sat_share7"]
def _demo(mask, label):
    d = panel[mask].dropna(subset=_f + ["target"]).copy()
    d["target_date"] = d["date"] + pd.Timedelta(days=1)
    d = d.sort_values("target_date"); i = int(0.85 * len(d))
    m = HistGradientBoostingRegressor(max_iter=200, learning_rate=0.05, max_depth=4, random_state=SEED).fit(d[_f].iloc[:i], d["target"].iloc[:i])
    print(f"{label:46s} rows={len(d):5d}  test R2 = {r2_score(d['target'].iloc[i:], m.predict(d[_f].iloc[i:])):.3f}")
_demo(panel["target"].notna(), "LEAKY: 'today' may be interpolated")
_demo(panel["target"].notna() & (panel["observed"] == 1), "CLEAN: 'today' must be a real observation")

The leaky version looks 2–3× better and is **not real skill**. Everything below uses the clean rule.

### Shared feature code
The feature builder is written to a module that both this notebook and the Streamlit app import, so training and serving can never drift apart.

In [ ]:
Path("roads.json").write_text(json.dumps(sorted(tr["road"].unique())))

In [ ]:
%%writefile mobility_features.py
# Shared feature code for Bengaluru Smart Mobility Intelligence (used by the notebook AND app.py)
import json, os, re
import numpy as np, pandas as pd

HERE = os.path.dirname(os.path.abspath(__file__))
ROADS = json.load(open(os.path.join(HERE, "roads.json")))

def slug(s):
    return re.sub(r"[^0-9a-zA-Z]+", "_", str(s)).strip("_").lower()

ROAD_COLS = ["road_" + slug(r) for r in ROADS]
OPS = ["traffic_volume", "average_speed", "travel_time_index", "congestion_level", "road_capacity_utilization",
       "incident_reports", "public_transport_usage", "traffic_signal_compliance", "parking_usage",
       "pedestrian_and_cyclist_count"]
WX = ["rain_sum", "temp_mean", "humidity_mean", "wind_max", "cloud_mean"]
WX_NEXT = [c + "_next" for c in WX]
LAGS = ["cong_lag1", "cong_roll3", "cong_roll7", "cong_std7", "sat_share7", "vol_roll7"]
CAL_N = ["dow_n_sin", "dow_n_cos", "weekend_n", "month_n", "monsoon_n"]      # calendar of the TARGET day (tomorrow)
CAL_T = ["dow_sin", "dow_cos", "weekend", "month", "monsoon"]                # calendar of TODAY

# Task A: forecast congestion tomorrow
A_BASE = OPS + ["roadwork"] + WX + WX_NEXT + CAL_N + ROAD_COLS
A_LAG = LAGS
A_ENG = ["pressure", "vol_x_cong", "speed_x_cong", "incident_rate", "weekend_n_x_vol", "roadwork_x_vol", "rain_next_x_vol"]
# Task B: estimate congestion now from its drivers (no congestion-derived inputs)
B_DRIVERS = ["traffic_volume", "incident_reports", "public_transport_usage", "traffic_signal_compliance",
             "parking_usage", "pedestrian_and_cyclist_count", "roadwork"] + WX + CAL_T + ROAD_COLS
B_ENG = ["incident_rate", "weekend_x_vol", "roadwork_x_vol", "rain_x_vol"]
B_SENSORS = ["average_speed", "travel_time_index", "road_capacity_utilization"]   # co-measured with congestion: reported, not deployed
CONGESTION_DERIVED = {"congestion_level", "vol_x_cong", "speed_x_cong"}           # must never appear in Task B inputs

def build_features(df):
    # df needs: date, road, the OPS columns, roadwork, WX, WX_NEXT and (for Task A) the LAGS columns
    d = df.copy()
    t = pd.to_datetime(d["date"]); n = t + pd.Timedelta(days=1)
    d["dow"] = t.dt.dayofweek
    d["dow_sin"] = np.sin(2 * np.pi * d["dow"] / 7); d["dow_cos"] = np.cos(2 * np.pi * d["dow"] / 7)
    d["weekend"] = (d["dow"] >= 5).astype(int); d["month"] = t.dt.month
    d["monsoon"] = t.dt.month.isin([6, 7, 8, 9]).astype(int)
    dn = n.dt.dayofweek
    d["dow_n_sin"] = np.sin(2 * np.pi * dn / 7); d["dow_n_cos"] = np.cos(2 * np.pi * dn / 7)
    d["weekend_n"] = (dn >= 5).astype(int); d["month_n"] = n.dt.month
    d["monsoon_n"] = n.dt.month.isin([6, 7, 8, 9]).astype(int)
    vol = d["traffic_volume"]
    d["pressure"] = vol * d["road_capacity_utilization"] / 1e5               # load x how full the road already is
    d["vol_x_cong"] = vol * d["congestion_level"] / 1e5
    d["speed_x_cong"] = d["average_speed"] * d["congestion_level"] / 100
    d["incident_rate"] = d["incident_reports"] / (vol / 1e4).clip(lower=1)    # incidents per 10k vehicles
    d["weekend_x_vol"] = d["weekend"] * vol / 1e4
    d["weekend_n_x_vol"] = d["weekend_n"] * vol / 1e4
    d["roadwork_x_vol"] = d["roadwork"] * vol / 1e4
    d["rain_x_vol"] = d["rain_sum"] * vol / 1e5
    d["rain_next_x_vol"] = d["rain_sum_next"] * vol / 1e5
    for r, c in zip(ROADS, ROAD_COLS):
        d[c] = (d["road"] == r).astype(int)
    return d

In [ ]:
import importlib, sys
sys.path.insert(0, PROJECT_DIR)
import mobility_features as mf
importlib.reload(mf)

# Task A table: today is a REAL observation, tomorrow is a REAL observation, and the 7-day history is complete
need = mf.LAGS + mf.WX + mf.WX_NEXT
df_fc = panel[(panel["observed"] == 1) & panel["target"].notna()].dropna(subset=need).copy()
df_fc["target_date"] = df_fc["date"] + pd.Timedelta(days=1)
df_fc = mf.build_features(df_fc).sort_values("target_date").reset_index(drop=True)
print("Task A rows:", len(df_fc), "| roads:", df_fc["road"].nunique(), "|", df_fc["target_date"].min().date(), "->", df_fc["target_date"].max().date())

# Task B table: every real observation with weather available; target = congestion of the same day
df_wi = panel[panel["observed"] == 1].dropna(subset=mf.WX).copy()
df_wi = mf.build_features(df_wi.assign(rain_sum_next=df_wi["rain_sum"], temp_mean_next=df_wi["temp_mean"]))
df_wi = df_wi.sort_values("date").reset_index(drop=True)
print("Task B rows:", len(df_wi))

## Phase 5 — Target distribution  *(your Step 2)*

In [ ]:
y_all = df_fc["target"]
fig, ax = plt.subplots(1, 3, figsize=(17, 4))
sns.histplot(y_all, bins=40, kde=True, ax=ax[0]); ax[0].set_title("Next-day congestion: histogram + KDE")
sns.boxplot(data=df_fc, y="road", x="target", ax=ax[1], orient="h"); ax[1].set_title("By road")
sns.ecdfplot(y_all, ax=ax[2]); ax[2].set_title("Cumulative distribution (note the jump at 100)")
plt.tight_layout(); savefig("target_distribution"); plt.show()

q1, q3 = y_all.quantile([.25, .75]); iqr = q3 - q1
print(f"skewness = {y_all.skew():.2f} | excess kurtosis = {y_all.kurt():.2f}")
print(f"exactly 100: {(y_all >= 99.9).mean():.1%} | below IQR fence ({q1 - 1.5 * iqr:.1f}): {(y_all < q1 - 1.5 * iqr).mean():.1%} | median {y_all.median():.1f}")

**Reading it:** the distribution is strongly **left-skewed with a spike at the ceiling of 100** — a censored variable, not a normal one. The "outliers" flagged by the IQR rule are simply low-congestion days, not errors, so nothing is removed.

**Does a target transformation help?** We test it properly in the next cell, *after* fixing the split (so nothing here peeks at the test set). Options: raw target; `log1p(100 − y)` (turns the ceiling spike into zeros); and a two-part "hurdle" model (probability of hitting 100 × expected value otherwise).

## Phase 6 — Is the train/test split appropriate?  *(your Step 6, moved forward on purpose)*
We fix the split **before** any supervised analysis so mutual information, importances and feature choices never see the test set.

Traffic is a time series: using tomorrow's neighbours to predict yesterday is unrealistic and lets the model memorise adjacent days. We therefore use a **chronological** split — the first 85 % of dates for development (model selection with expanding-window `TimeSeriesSplit` cross-validation) and the last 15 % as an untouched test set.

In [ ]:
cut = df_fc["target_date"].iloc[int(0.85 * len(df_fc))]
dev = df_fc[df_fc["target_date"] < cut].copy()
test = df_fc[df_fc["target_date"] >= cut].copy()
print(f"development: {len(dev)} rows ({dev['target_date'].min().date()} -> {dev['target_date'].max().date()})")
print(f"test       : {len(test)} rows ({test['target_date'].min().date()} -> {test['target_date'].max().date()})")

# random vs chronological validation on the same model (development data only)
probe = mf.A_BASE + mf.A_LAG
hgb = HistGradientBoostingRegressor(max_iter=200, learning_rate=0.05, max_depth=4, random_state=SEED)
r_rand = cross_val_score(hgb, dev[probe], dev["target"], cv=KFold(5, shuffle=True, random_state=SEED), scoring="r2")
r_time = cross_val_score(hgb, dev[probe], dev["target"], cv=TimeSeriesSplit(5), scoring="r2")
print(f"\nrandom 5-fold CV R2        : {r_rand.mean():.3f} ± {r_rand.std():.3f}")
print(f"chronological 5-fold CV R2 : {r_time.mean():.3f} ± {r_time.std():.3f}")

**Decision:** keep the chronological split. If random-fold R² is not lower here, that is because day-to-day dependence in this data is weak — the chronological split is still the honest design for a forecasting product and costs nothing.

In [ ]:
# ---- target-transformation experiment (development data, chronological CV) ----
Xp, yp = dev[probe], dev["target"]
def cv_pred_r2(fit_predict):
    s = []
    for a, b in TimeSeriesSplit(5).split(Xp):
        s.append(r2_score(yp.iloc[b], np.clip(fit_predict(Xp.iloc[a], yp.iloc[a], Xp.iloc[b]), 0, 100)))
    return np.mean(s), np.std(s)

def raw_fp(Xa, ya, Xb):
    return make_pipeline(StandardScaler(), Ridge(50)).fit(Xa, ya).predict(Xb)
def log_fp(Xa, ya, Xb):
    m = make_pipeline(StandardScaler(), Ridge(50)).fit(Xa, np.log1p(100 - ya))
    return 100 - np.expm1(m.predict(Xb))
def hurdle_fp(Xa, ya, Xb):
    sat = (ya >= 99.9).astype(int)
    clf = make_pipeline(StandardScaler(), LogisticRegression(C=0.3, max_iter=500)).fit(Xa, sat)
    reg = make_pipeline(StandardScaler(), Ridge(50)).fit(Xa[sat == 0], ya[sat == 0])
    p = clf.predict_proba(Xb)[:, 1]
    return p * 100 + (1 - p) * np.clip(reg.predict(Xb), 0, 100)

rows = [(n, *cv_pred_r2(f)) for n, f in [("Raw target (Ridge)", raw_fp), ("log1p(100 - y) (Ridge)", log_fp), ("Hurdle: P(=100) x 100 + (1-P) x E[y | <100]", hurdle_fp)]]
display(pd.DataFrame(rows, columns=["Target treatment", "CV R2 mean", "CV R2 std"]).round(3))

**Decision:** a transformation is only worth its complexity if it clearly beats the raw target. Because the censoring is real, we additionally keep a **classifier for "will it saturate (=100)?"** in Phase 11 — that is the more natural framing of what the data can say about tomorrow.

## Phase 7 — Feature–target relationships  *(your Step 3)*
Pearson only sees straight lines. We add Spearman (monotonic), **mutual information** (any dependence) and **tree importance**, all computed on the development set only, and look at binned relationships instead of raw scatter (the target is censored, so raw scatter is a blob).

In [ ]:
num_feats = (mf.OPS + ["roadwork"] + mf.WX + mf.WX_NEXT + mf.LAGS + ["dow_n_sin", "dow_n_cos", "weekend_n", "month_n", "monsoon_n"])
Xd, yd = dev[num_feats], dev["target"]
rf_probe = RandomForestRegressor(300, min_samples_leaf=10, max_features=0.5, n_jobs=-1, random_state=SEED).fit(Xd, yd)
rel = pd.DataFrame({
    "pearson": Xd.corrwith(yd), "spearman": Xd.corrwith(yd, method="spearman"),
    "mutual_info": pd.Series(mutual_info_regression(Xd, yd, random_state=SEED), index=num_feats),
    "rf_importance": pd.Series(rf_probe.feature_importances_, index=num_feats)}).sort_values("mutual_info", ascending=False)
display(rel.round(3).head(14))

gm = dev.groupby("road")["target"].transform("mean")
eta2 = ((gm - yd.mean()) ** 2).sum() / ((yd - yd.mean()) ** 2).sum()
print(f"\nShare of next-day congestion variance explained by WHICH ROAD it is (eta^2): {eta2:.3f}")

top = rel.index[:4].tolist() + ["cong_roll7"]
fig, ax = plt.subplots(1, 5, figsize=(21, 3.8))
for a, f in zip(ax, dict.fromkeys(top)):
    b = pd.qcut(dev[f], 10, duplicates="drop")
    s = dev.groupby(b)["target"].agg(["mean", "sem"]).reset_index(drop=True)
    a.errorbar(range(len(s)), s["mean"], yerr=1.96 * s["sem"], marker="o"); a.set_title(f, fontsize=10)
    a.set_xlabel("decile of feature"); a.set_ylabel("mean next-day congestion")
plt.tight_layout(); savefig("feature_relationships"); plt.show()

**Reading it.** Features with a rank-based or mutual-information signal that Pearson missed would show up as high `mutual_info` with low `pearson`; compare the columns. The binned plots show how flat or steep each relationship really is, with 95 % bands — if the band covers the whole range, the feature carries no usable signal on its own.

**Decision:** no feature is dropped because of a low Pearson value. Selection is made later by cross-validated performance (Phase 8), not by single correlations.

## Phase 7b — Multicollinearity  *(your Step 4)*

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

def vif_table(df, cols):
    X = add_constant(df[cols].astype(float))
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        return _vif(X, cols)

def _vif(X, cols):
    return pd.Series([variance_inflation_factor(X.values, i + 1) for i in range(len(cols))], index=cols).sort_values(ascending=False)

with_env = vif_table(tr, mf.OPS + ["roadwork", "environmental_impact"])
print("VIF WITH Environmental Impact (inf = perfect duplicate):"); print(with_env.round(1).head(4).to_string())
core = mf.OPS + ["roadwork"] + mf.WX
print("\nVIF after dropping it (rule of thumb: > 5 worth a look, > 10 problematic):"); print(vif_table(dev, core).round(2).head(8).to_string())

corr = dev[core].corr()
pairs = corr.where(np.triu(np.ones(corr.shape), 1).astype(bool)).stack()
print("\nPairs with |r| > 0.5:"); print(pairs[pairs.abs() > 0.5].round(2).to_string() or "none")
plt.figure(figsize=(9, 7)); sns.heatmap(corr, cmap="RdBu_r", center=0, annot=True, fmt=".2f", annot_kws={"size": 7}); plt.title("Correlation matrix (development set)")
plt.tight_layout(); savefig("correlation_matrix"); plt.show()

**Decision:** `Environmental Impact` is removed (exact duplicate of `Traffic Volume`, VIF → ∞). `congestion_level` and `road_capacity_utilization` overlap strongly with each other and with volume and pedestrians (VIF around 8–11) — that is real traffic physics (a full road is a congested road), not an error. We do **not** delete variables just because they overlap: regularised and tree models handle it, and the ablation below decides what is worth keeping.

## Phase 8 — Feature engineering and selection  *(your Step 5)*
Every engineered feature has a mobility reason:

| Feature | Reasoning |
|---|---|
| `pressure` = volume × capacity utilisation | demand on a road that is already full |
| `vol_x_cong` , `speed_x_cong` | load interacting with the current congestion state |
| `incident_rate` | incidents per 10 000 vehicles — risk normalised for exposure |
| `weekend_n_x_vol`, `roadwork_x_vol` | weekday/weekend and roadwork effects scale with traffic load |
| `rain_next_x_vol` | tomorrow's rain matters more when volume is high |
| `cong_lag1`, `cong_roll3/7`, `cong_std7`, `sat_share7`, `vol_roll7` | recent history of the road |
| calendar of **tomorrow**, monsoon flag (Jun–Sep) | known in advance |

We **test** whether each group actually helps with chronological cross-validation on the development set, and keep the best-performing set.

In [ ]:
GROUPS = {"road identity only": mf.ROAD_COLS, "base": mf.A_BASE, "base + lags": mf.A_BASE + mf.A_LAG,
          "base + engineered": mf.A_BASE + mf.A_ENG, "base + lags + engineered": mf.A_BASE + mf.A_LAG + mf.A_ENG}
probe_models = {"Ridge": make_pipeline(StandardScaler(), Ridge(50)),
                "ExtraTrees": ExtraTreesRegressor(150, min_samples_leaf=10, max_features=0.5, n_jobs=-1, random_state=SEED),
                "HistGB": HistGradientBoostingRegressor(max_iter=150, learning_rate=0.05, max_depth=3, random_state=SEED)}
abl = pd.DataFrame({g: {m: cross_val_score(est, dev[cols], dev["target"], cv=TimeSeriesSplit(5), scoring="r2").mean()
                        for m, est in probe_models.items()} for g, cols in GROUPS.items()}).T
abl["mean"] = abl.mean(axis=1)
display(abl.round(3))
best_group = abl["mean"].idxmax()
FEATS_A = GROUPS[best_group]
print(f"\nSelected feature set for Task A: '{best_group}' ({len(FEATS_A)} features)")
assert not any(c in FEATS_A for c in ["target", "target_date"]), "leakage guard"

X_train, y_train = dev[FEATS_A], dev["target"]
X_test, y_test = test[FEATS_A], test["target"]

**Reading it.** If "road identity only" is within a hair of the richer sets, that tells us something important and honest: *knowing which road it is explains almost everything the data can say about tomorrow*, and the extra features add little. The ablation, not the number of features, decides what goes in.

## Phase 9 — Model toolkit  *(your Steps 7–9)*
Models: Ridge, Random Forest, Extra Trees, Gradient Boosting, HistGradientBoosting, XGBoost, LightGBM, CatBoost — plus three baselines (global mean, per-road mean, persistence). All are scored with **expanding-window time-series cross-validation** (R², MAE, RMSE); R² is the primary metric.

In [ ]:
class RoadMeanRegressor(BaseEstimator, RegressorMixin):
    """Baseline: predict each road's historical mean."""
    def fit(self, X, y):
        idx = X[mf.ROAD_COLS].values.argmax(1)
        self.means_ = pd.Series(np.asarray(y)).groupby(idx).mean(); self.global_ = float(np.mean(y)); return self
    def predict(self, X):
        return pd.Series(X[mf.ROAD_COLS].values.argmax(1)).map(self.means_).fillna(self.global_).values

class PersistenceRegressor(BaseEstimator, RegressorMixin):
    """Baseline: tomorrow = today (looks today's congestion up by row index, so it works with any feature set)."""
    def __init__(self, source=None):
        self.source = source
    def fit(self, X, y): return self
    def predict(self, X): return self.source.loc[X.index].values

def model_zoo():
    return {
        "Ridge": make_pipeline(StandardScaler(), Ridge(alpha=50)),
        "Random Forest": RandomForestRegressor(n_estimators=300, min_samples_leaf=5, max_features=0.5, n_jobs=1, random_state=SEED),
        "Extra Trees": ExtraTreesRegressor(n_estimators=300, min_samples_leaf=5, max_features=0.5, n_jobs=1, random_state=SEED),
        "Gradient Boosting": GradientBoostingRegressor(n_estimators=200, learning_rate=0.05, max_depth=3, subsample=0.8, random_state=SEED),
        "HistGradientBoosting": HistGradientBoostingRegressor(max_iter=300, learning_rate=0.04, max_depth=4, l2_regularization=1.0, random_state=SEED),
        "XGBoost": XGBRegressor(n_estimators=300, learning_rate=0.03, max_depth=3, subsample=0.8, colsample_bytree=0.8, min_child_weight=5, n_jobs=1, random_state=SEED),
        "LightGBM": LGBMRegressor(n_estimators=300, learning_rate=0.03, num_leaves=15, min_child_samples=20, subsample=0.8, subsample_freq=1, colsample_bytree=0.8, n_jobs=1, verbose=-1, random_state=SEED),
        "CatBoost": CatBoostRegressor(iterations=300, learning_rate=0.05, depth=4, verbose=0, random_seed=SEED, thread_count=1, allow_writing_files=False)}

GRIDS = {   # meaningful parameters only (your Step 8); random search over these, time-series CV
    "Ridge": {"ridge__alpha": [1, 10, 50, 200, 1000, 5000]},
    "Random Forest": {"n_estimators": [200, 400], "max_depth": [4, 6, 8, None], "min_samples_split": [2, 10, 30],
                      "min_samples_leaf": [3, 5, 10, 20], "max_features": [0.3, 0.5, 0.8]},
    "Gradient Boosting": {"n_estimators": [100, 200, 300], "learning_rate": [0.01, 0.03, 0.05, 0.1], "max_depth": [2, 3, 4],
                          "min_samples_split": [2, 10, 30], "min_samples_leaf": [5, 10, 20], "subsample": [0.6, 0.8, 1.0]},
    "HistGradientBoosting": {"learning_rate": [0.02, 0.04, 0.08], "max_depth": [2, 3, 4, 6], "min_samples_leaf": [10, 20, 50],
                             "l2_regularization": [0.0, 1.0, 10.0], "max_iter": [150, 300]},
    "XGBoost": {"n_estimators": [150, 300, 500], "learning_rate": [0.01, 0.03, 0.06], "max_depth": [2, 3, 4, 5],
                "min_child_weight": [1, 5, 10, 20], "subsample": [0.6, 0.8, 1.0], "colsample_bytree": [0.5, 0.8, 1.0]},
    "LightGBM": {"n_estimators": [150, 300, 500], "learning_rate": [0.01, 0.03, 0.06], "num_leaves": [7, 15, 31],
                 "min_child_samples": [10, 20, 50], "subsample": [0.6, 0.8, 1.0], "colsample_bytree": [0.5, 0.8, 1.0], "reg_lambda": [0, 1, 10]}}
TUNE = list(GRIDS)
COMPLEXITY = {"Ridge": 1, "Random Forest": 3, "Extra Trees": 3, "Gradient Boosting": 4, "HistGradientBoosting": 4,
              "XGBoost": 4, "LightGBM": 4, "CatBoost": 4}

def cv_summary(model, X, y, k=5):
    r = cross_validate(clone(model), X, y, cv=TimeSeriesSplit(k), n_jobs=1,
                       scoring={"r2": "r2", "mae": "neg_mean_absolute_error", "rmse": "neg_root_mean_squared_error"})
    return dict(cv_r2=r["test_r2"].mean(), cv_r2_std=r["test_r2"].std(), cv_r2_se=r["test_r2"].std() / np.sqrt(k),
                cv_mae=-r["test_mae"].mean(), cv_rmse=-r["test_rmse"].mean())

def test_summary(model, Xtr, ytr, Xte, yte):
    m = clone(model).fit(Xtr, ytr); p = np.clip(m.predict(Xte), 0, 100)
    return dict(MAE=mean_absolute_error(yte, p), RMSE=float(np.sqrt(mean_squared_error(yte, p))), R2=r2_score(yte, p)), m

def tune(name, model, X, y, n_iter=N_ITER):
    size = math.prod(len(v) for v in GRIDS[name].values())
    s = RandomizedSearchCV(clone(model), GRIDS[name], n_iter=min(n_iter, size), cv=TimeSeriesSplit(3), scoring="r2",
                           random_state=SEED, n_jobs=-1, refit=True).fit(X, y)
    return s.best_estimator_, s.best_params_

def run_comparison(X_tr, y_tr, X_te, y_te, baselines, tune_names=TUNE):
    rows, fitted, params = [], {}, {}
    def add(name, base, typ, est):
        cvs = cv_summary(est, X_tr, y_tr); ts, m = test_summary(est, X_tr, y_tr, X_te, y_te)
        rows.append({"Model": name, "Base": base, "Type": typ, **cvs, "Test MAE": ts["MAE"], "Test RMSE": ts["RMSE"], "Test R2": ts["R2"]})
        fitted[name] = m
    for n, e in baselines.items(): add(n, n, "baseline", e)
    zoo = model_zoo()
    for n, e in zoo.items(): add(n, n, "default", e)
    for n in tune_names:
        best, p = tune(n, zoo[n], X_tr, y_tr); params[n] = p; add(f"{n} (tuned)", n, "tuned", best)
    return pd.DataFrame(rows), fitted, params

def select_final(res):
    """Highest CV R2 within one standard error, then the SIMPLEST model among those (ties -> better CV R2)."""
    cand = res[res["Type"] != "baseline"].copy()
    best = cand.loc[cand["cv_r2"].idxmax()]
    elig = cand[cand["cv_r2"] >= best["cv_r2"] - best["cv_r2_se"]].copy()
    elig["complexity"] = elig["Base"].map(COMPLEXITY)
    pick = elig.sort_values(["complexity", "cv_r2"], ascending=[True, False]).iloc[0]
    return pick, best, elig

def show_table(res, cols=("Model", "Type", "cv_r2", "cv_r2_std", "Test MAE", "Test RMSE", "Test R2")):
    t = res.sort_values("cv_r2", ascending=False)[list(cols)].rename(columns={"cv_r2": "CV R2", "cv_r2_std": "CV R2 std"})
    display(t.round(3).reset_index(drop=True))

def plot_comparison(res, title, fname):
    r = res.sort_values("cv_r2")
    fig, ax = plt.subplots(figsize=(9, 0.38 * len(r) + 1.5))
    ax.barh(r["Model"], r["cv_r2"], xerr=r["cv_r2_std"], color="#4C72B0", alpha=.8, label="CV R2 (± std)")
    ax.scatter(r["Test R2"], r["Model"], color="#DD8452", zorder=3, label="Test R2")
    ax.axvline(0, color="k", lw=.8); ax.set_title(title); ax.legend(loc="lower right"); plt.tight_layout(); savefig(fname); plt.show()

## Phase 10 — Task A: tomorrow's congestion — compare, tune, cross-validate  *(your Steps 7–10)*

In [ ]:
baselines_A = {"Mean (no model)": DummyRegressor(), "Per-road mean": RoadMeanRegressor(), "Persistence (tomorrow = today)": PersistenceRegressor(source=df_fc["congestion_level"])}
res_A, fitted_A, params_A = run_comparison(X_train, y_train, X_test, y_test, baselines_A)
show_table(res_A)
plot_comparison(res_A, "Task A — tomorrow's congestion: CV R2 vs test R2", "model_comparison_A")

**Step 10 table (test set):** `Test MAE | Test RMSE | Test R2` above, with the cross-validated R² and its spread. Note that tuned CV scores are slightly optimistic (the same folds were used to choose parameters) — the test column is the unbiased one.

## Phase 11 — Final model selection  *(your Step 11)*
Rule, stated *before* looking at the answer: take the best cross-validated R², allow every model within **one standard error** of it, and pick the **simplest** of those (interpretable and cheap to deploy). Test scores are reported, never used to choose.

In [ ]:
pick_A, best_A, elig_A = select_final(res_A)
print(f"Best CV R2: {best_A['Model']} = {best_A['cv_r2']:.3f} (se {best_A['cv_r2_se']:.3f})")
print("Within one standard error:", ", ".join(elig_A["Model"]))
print(f"\n>>> FINAL MODEL (Task A): {pick_A['Model']} | CV R2 {pick_A['cv_r2']:.3f} | test R2 {pick_A['Test R2']:.3f} | test MAE {pick_A['Test MAE']:.2f}")
rm = res_A.loc[res_A["Model"] == "Per-road mean"].iloc[0]
print(f"Reference: per-road mean test R2 = {rm['Test R2']:.3f}; persistence test R2 = {res_A.loc[res_A['Model'].str.startswith('Persistence'), 'Test R2'].iloc[0]:.3f}")
final_A_name = pick_A["Model"]; final_A = fitted_A[final_A_name]

**How to read the result honestly.** Tomorrow's congestion on a road is mostly *its usual level plus day-to-day noise*. If the final model is only a little above the "per-road mean" baseline, that is the data's real ceiling — not a modelling failure — and we say so rather than chase R² with leaky features. The more useful product from this data is therefore (1) a calibrated **probability that tomorrow saturates** and (2) the **what-if estimator** in Task B.

### Task A2 — "Severe congestion tomorrow?" alert classifier
Half the target sits at the ceiling, so *will it saturate (=100)?* is a natural, decision-ready question. Evaluated with AUC (ranking quality) and Brier score (calibration), cross-validated chronologically.

In [ ]:
sat_train, sat_test = (y_train >= 99.9).astype(int), (y_test >= 99.9).astype(int)

# which feature group suits the classifier? (logistic regression, chronological CV AUC; simplest set within one standard error)
lr_probe = make_pipeline(StandardScaler(), LogisticRegression(C=0.3, max_iter=1000))
auc_g = {}
for g, cols in GROUPS.items():
    sc = cross_val_score(lr_probe, dev[cols], sat_train, cv=TimeSeriesSplit(5), scoring="roc_auc")
    auc_g[g] = dict(cv_auc=sc.mean(), se=sc.std() / np.sqrt(5), n_features=len(cols))
auc_g = pd.DataFrame(auc_g).T
display(auc_g.round(3))
okg = auc_g[auc_g["cv_auc"] >= auc_g["cv_auc"].max() - auc_g.loc[auc_g["cv_auc"].idxmax(), "se"]]
best_group_alert = okg["n_features"].astype(float).idxmin()
FEATS_ALERT = GROUPS[best_group_alert]
print(f"Alert-classifier feature set: '{best_group_alert}' ({len(FEATS_ALERT)} features)")
Xa_train, Xa_test = dev[FEATS_ALERT], test[FEATS_ALERT]

clfs = {"Logistic regression": make_pipeline(StandardScaler(), LogisticRegression(C=0.3, max_iter=1000)),
        "HistGB classifier": HistGradientBoostingClassifier(max_iter=200, learning_rate=0.04, max_depth=3, l2_regularization=1.0, random_state=SEED),
        "LightGBM classifier": LGBMClassifier(n_estimators=250, learning_rate=0.03, num_leaves=8, min_child_samples=30, subsample=0.8, subsample_freq=1, colsample_bytree=0.8, n_jobs=1, verbose=-1, random_state=SEED)}
rows = []
for n, c in clfs.items():
    cv = cross_val_score(c, Xa_train, sat_train, cv=TimeSeriesSplit(5), scoring="roc_auc")
    m = clone(c).fit(Xa_train, sat_train); p = m.predict_proba(Xa_test)[:, 1]; pred = (p >= 0.5).astype(int)
    rows.append({"Model": n, "CV AUC": cv.mean(), "CV AUC se": cv.std() / np.sqrt(5), "Test AUC": roc_auc_score(sat_test, p),
                 "Accuracy": accuracy_score(sat_test, pred), "Precision": precision_score(sat_test, pred), "Recall": recall_score(sat_test, pred),
                 "F1": f1_score(sat_test, pred), "Brier": brier_score_loss(sat_test, p), "_m": m})
clf_res = pd.DataFrame(rows)
display(clf_res.drop(columns="_m").round(3))
print("Base rate of 'saturated tomorrow' in test:", round(sat_test.mean(), 3), "| Brier of always predicting the base rate:", round(brier_score_loss(sat_test, np.full(len(sat_test), sat_train.mean())), 3))
b = clf_res.loc[clf_res["CV AUC"].idxmax()]; el = clf_res[clf_res["CV AUC"] >= b["CV AUC"] - b["CV AUC se"]]
alert_name = el.iloc[0]["Model"]      # first listed = simplest
alert_model = clf_res.loc[clf_res["Model"] == alert_name, "_m"].iloc[0]
print(f">>> ALERT MODEL: {alert_name}")

## Phase 12 — Task B: what-if congestion estimator
*"What congestion should we expect under these conditions?"* Inputs are the **drivers**: traffic volume, incidents, roadwork, public-transport usage, signal compliance, parking, pedestrians, weather, calendar and the road. The three sensor readings co-measured with congestion (`average_speed`, `travel_time_index`, `road_capacity_utilization`) are **not** inputs: they are outcomes of the same congestion, so using them would answer a different (and circular) question. We report what they would add, for transparency.

In [ ]:
cutb = df_wi["date"].iloc[int(0.85 * len(df_wi))]
devb, testb = df_wi[df_wi["date"] < cutb].copy(), df_wi[df_wi["date"] >= cutb].copy()
GROUPS_B = {"drivers": mf.B_DRIVERS, "drivers + engineered": mf.B_DRIVERS + mf.B_ENG,
            "[reference only] drivers + engineered + sensors": mf.B_DRIVERS + mf.B_ENG + mf.B_SENSORS}
ablB = pd.DataFrame({g: {m: cross_val_score(est, devb[cols], devb["congestion_level"], cv=TimeSeriesSplit(5), scoring="r2").mean()
                         for m, est in probe_models.items()} for g, cols in GROUPS_B.items()}).T
ablB["mean"] = ablB.mean(axis=1)
display(ablB.round(3))
deployable = ablB.drop(index=[i for i in ablB.index if i.startswith("[reference")])
best_group_B = deployable["mean"].idxmax(); FEATS_B = GROUPS_B[best_group_B]
assert not (set(FEATS_B) & mf.CONGESTION_DERIVED), "Task B must not use congestion-derived inputs"
print(f"\nSelected feature set for Task B: '{best_group_B}' ({len(FEATS_B)} features)")
Xb_train, yb_train = devb[FEATS_B], devb["congestion_level"]
Xb_test, yb_test = testb[FEATS_B], testb["congestion_level"]
print(f"development {len(devb)} rows | test {len(testb)} rows ({testb['date'].min().date()} -> {testb['date'].max().date()})")

In [ ]:
baselines_B = {"Mean (no model)": DummyRegressor(), "Per-road mean": RoadMeanRegressor()}
res_B, fitted_B, params_B = run_comparison(Xb_train, yb_train, Xb_test, yb_test, baselines_B)
show_table(res_B)
plot_comparison(res_B, "Task B — what-if congestion estimator: CV R2 vs test R2", "model_comparison_B")
pick_B, best_B, elig_B = select_final(res_B)
print(f"\nBest CV R2: {best_B['Model']} = {best_B['cv_r2']:.3f} | within 1 se: {', '.join(elig_B['Model'])}")
print(f">>> FINAL MODEL (Task B): {pick_B['Model']} | CV R2 {pick_B['cv_r2']:.3f} | test R2 {pick_B['Test R2']:.3f} | test MAE {pick_B['Test MAE']:.2f} | RMSE {pick_B['Test RMSE']:.2f}")
final_B_name = pick_B["Model"]; final_B = fitted_B[final_B_name]

**Why this one reaches a high R² — and the caveat.** Congestion at a moment is largely *determined* by the load at that moment, and the relationship is non-linear (it saturates at 100), which is exactly what trees capture and a straight line (Ridge, ≈0.86) cannot. Phase 14 shows that in this dataset **traffic volume alone** explains almost all of it, so read the score as "congestion ≈ a saturating function of volume", not as a rich multi-factor model. It is an **estimator for planning and what-if analysis**, not a forecast; Task A is the forecast. Both are reported with their own honest ceilings.

## Phase 12b — Deep-learning comparison: GRU and LSTM
Does a sequence model find a temporal pattern the simpler models miss? Each sample is the previous 7 days for one road → tomorrow's congestion. The last day of every window must be a **real** observation (same leakage rule as before), scaling comes from training windows only, the split is chronological, and the last 15 % of the development windows are used for early stopping. Compared against the per-road mean on exactly the same windows.

In [ ]:
import torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(SEED)
SEQ_LEN = 7
SEQ_FEATS = mf.OPS + ["roadwork"] + mf.WX + mf.WX_NEXT + ["dow_n_sin", "dow_n_cos", "weekend_n"]
pf = mf.build_features(panel)

def make_windows(pf):
    X, R, Y, TD = [], [], [], []
    for road, g in pf.groupby("road"):
        g = g.sort_values("date"); arr = g[SEQ_FEATS].to_numpy(float); ob = g["observed"].to_numpy(); y = g["target"].to_numpy(float)
        td = (g["date"] + pd.Timedelta(days=1)).to_numpy(); rid = mf.ROADS.index(road)
        for i in range(SEQ_LEN - 1, len(g)):
            if ob[i] != 1 or np.isnan(y[i]): continue
            w_ = arr[i - SEQ_LEN + 1: i + 1]
            if np.isnan(w_).any(): continue
            X.append(w_); R.append(rid); Y.append(y[i]); TD.append(td[i])
    return np.asarray(X, np.float32), np.asarray(R), np.asarray(Y, np.float32), np.asarray(TD)

Xw, Rw, Yw, TDw = make_windows(pf)
is_test = TDw >= np.datetime64(cut)
dev_idx = np.where(~is_test)[0]; te_idx = np.where(is_test)[0]
order_ = dev_idx[np.argsort(TDw[dev_idx])]; n_val = int(0.15 * len(order_))
va_idx, tr_idx = order_[-n_val:], order_[:-n_val]
Fn = Xw.shape[2]; mu = Xw[tr_idx].reshape(-1, Fn).mean(0); sd = Xw[tr_idx].reshape(-1, Fn).std(0) + 1e-6
Xs = ((Xw - mu) / sd).astype(np.float32)
print("windows:", len(Xw), "| train/val/test:", len(tr_idx), len(va_idx), len(te_idx))

class SeqModel(nn.Module):
    def __init__(self, n_feat, n_roads, cell="gru", hidden=48, emb=6, drop=0.3):
        super().__init__()
        self.emb = nn.Embedding(n_roads, emb)
        self.rnn = (nn.GRU if cell == "gru" else nn.LSTM)(n_feat + emb, hidden, batch_first=True)
        self.head = nn.Sequential(nn.Dropout(drop), nn.Linear(hidden, 24), nn.ReLU(), nn.Linear(24, 1))
    def forward(self, x, road):
        e = self.emb(road).unsqueeze(1).expand(-1, x.size(1), -1)
        out, _ = self.rnn(torch.cat([x, e], -1)); return self.head(out[:, -1]).squeeze(-1)

@torch.no_grad()
def seq_predict(model, idx):
    model.eval(); xb = torch.tensor(Xs[idx], device=device); rb = torch.tensor(Rw[idx], dtype=torch.long, device=device)
    return np.clip(model(xb, rb).cpu().numpy() * 100, 0, 100)

def train_seq(cell, epochs=60, patience=8):
    m = SeqModel(Fn, len(mf.ROADS), cell).to(device); opt = torch.optim.AdamW(m.parameters(), lr=1e-3, weight_decay=1e-3); lossf = nn.MSELoss()
    dl = DataLoader(TensorDataset(torch.tensor(Xs[tr_idx]), torch.tensor(Rw[tr_idx], dtype=torch.long), torch.tensor(Yw[tr_idx] / 100)), batch_size=64, shuffle=True)
    best, state, bad = 1e9, None, 0
    for ep in range(epochs):
        m.train()
        for xb, rb, yb in dl:
            xb, rb, yb = xb.to(device), rb.to(device), yb.to(device); opt.zero_grad(); lossf(m(xb, rb), yb).backward(); nn.utils.clip_grad_norm_(m.parameters(), 1.0); opt.step()
        v = np.sqrt(mean_squared_error(Yw[va_idx], seq_predict(m, va_idx)))
        if v < best - 1e-4: best, state, bad = v, copy.deepcopy(m.state_dict()), 0
        else:
            bad += 1
            if bad >= patience: break
    m.load_state_dict(state); return m

rows = []
rm_ = pd.Series(Yw[tr_idx]).groupby(Rw[tr_idx]).mean(); base_pred = pd.Series(Rw[te_idx]).map(rm_).fillna(Yw[tr_idx].mean()).values
rows.append(dict(Model="Per-road mean", MAE=mean_absolute_error(Yw[te_idx], base_pred), RMSE=np.sqrt(mean_squared_error(Yw[te_idx], base_pred)), R2=r2_score(Yw[te_idx], base_pred)))
for cell_ in ["gru", "lstm"]:
    net = train_seq(cell_); p = seq_predict(net, te_idx)
    rows.append(dict(Model=cell_.upper(), MAE=mean_absolute_error(Yw[te_idx], p), RMSE=np.sqrt(mean_squared_error(Yw[te_idx], p)), R2=r2_score(Yw[te_idx], p)))
seq_res = pd.DataFrame(rows); display(seq_res.round(3))
print("If GRU/LSTM do not clearly beat the per-road mean, deep learning adds nothing here: the signal is the road itself, not a temporal pattern.")

## Phase 13 — Final evaluation, residual analysis and error analysis
The final models are scored **once** on the untouched, chronologically later test set.

In [ ]:
from scipy import stats as sps
from sklearn.calibration import calibration_curve
from sklearn.metrics import roc_curve

def final_evaluation(model, Xte, yte, meta, date_col, title, fname):
    pred = np.clip(model.predict(Xte), 0, 100); res = yte.values - pred
    m = dict(MAE=mean_absolute_error(yte, pred), RMSE=float(np.sqrt(mean_squared_error(yte, pred))), R2=r2_score(yte, pred))
    fig, ax = plt.subplots(2, 2, figsize=(12, 9))
    ax[0, 0].scatter(yte, pred, s=10, alpha=.35); ax[0, 0].plot([0, 100], [0, 100], "r--"); ax[0, 0].set_xlabel("actual"); ax[0, 0].set_ylabel("predicted"); ax[0, 0].set_title("Actual vs predicted")
    ax[0, 1].scatter(pred, res, s=10, alpha=.35); ax[0, 1].axhline(0, color="r", ls="--"); ax[0, 1].set_xlabel("predicted"); ax[0, 1].set_ylabel("residual"); ax[0, 1].set_title("Residuals vs predicted")
    sns.histplot(res, kde=True, ax=ax[1, 0], bins=40); ax[1, 0].set_title("Residual distribution")
    sps.probplot(res, plot=ax[1, 1]); ax[1, 1].set_title("Residual Q-Q plot")
    fig.suptitle(f"{title}  |  R2 {m['R2']:.3f}  MAE {m['MAE']:.2f}  RMSE {m['RMSE']:.2f}", fontweight="bold")
    plt.tight_layout(); savefig(fname); plt.show()
    e = meta[["road", date_col]].copy(); e["abs_err"] = np.abs(res); e["resid"] = res; e["actual"] = yte.values
    print(f"bias (mean residual) = {res.mean():+.2f} | residual skew = {sps.skew(res):+.2f}")
    by_road = e.groupby("road").agg(n=("abs_err", "size"), MAE=("abs_err", "mean"), bias=("resid", "mean")).sort_values("MAE", ascending=False)
    by_dow = e.groupby(e[date_col].dt.day_name()).agg(MAE=("abs_err", "mean"), bias=("resid", "mean"))
    by_level = e.groupby(pd.cut(e["actual"], [-.1, 50, 75, 99.9, 100.1], labels=["<50", "50-75", "75-<100", "=100"])).agg(n=("abs_err", "size"), MAE=("abs_err", "mean"), bias=("resid", "mean"))
    print("\nWhere the model is weakest — by road (top 5):"); display(by_road.head(5).round(2))
    print("By actual congestion level:"); display(by_level.round(2))
    return m, by_road, by_level, by_dow

mA, byroad_A, bylevel_A, bydow_A = final_evaluation(final_A, X_test, y_test, test, "target_date", f"Task A — {final_A_name}: tomorrow's congestion", "eval_A")

**Error analysis, Task A.** Look at the "by actual level" table: a model that predicts each road's typical level will *over-predict* when tomorrow is light and *under-predict* when it saturates, so errors are largest at the extremes — that is the censoring and the day-to-day noise at work, not a bug. The per-road table shows which corridors are hardest to forecast.

In [ ]:
# alert classifier: ROC and calibration
p_alert = alert_model.predict_proba(Xa_test)[:, 1]
fpr, tpr, _ = roc_curve(sat_test, p_alert); frac, mean_p = calibration_curve(sat_test, p_alert, n_bins=8, strategy="quantile")
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(fpr, tpr); ax[0].plot([0, 1], [0, 1], "k--"); ax[0].set_title(f"{alert_name}: ROC (AUC {roc_auc_score(sat_test, p_alert):.2f})")
ax[1].plot(mean_p, frac, "o-"); ax[1].plot([0, 1], [0, 1], "k--"); ax[1].set_xlabel("predicted probability"); ax[1].set_ylabel("observed share"); ax[1].set_title("Calibration (closer to the diagonal = trustworthy probabilities)")
plt.tight_layout(); savefig("eval_alert"); plt.show()

In [ ]:
mB, byroad_B, bylevel_B, bydow_B = final_evaluation(final_B, Xb_test, yb_test, testb, "date", f"Task B — {final_B_name}: what-if congestion estimate", "eval_B")

**Error analysis, Task B.** Check the by-level table for the saturated class (`=100`): the model cannot distinguish "100" from "far above 100", so errors there are structurally limited. If errors are concentrated on a few roads, those are the roads whose capacity behaviour the features describe least well — candidates for extra sensors or road-specific models.

## Phase 14 — Explainable AI  *(permutation importance, then SHAP)*
Permutation importance asks "how much does the score drop if this feature is scrambled?" and is model-agnostic. SHAP then shows the *direction*: which feature values push the prediction up or down. SHAP is used here because the interesting question — *does more volume / an incident / roadwork raise congestion, and by how much?* — is a directional one.

In [ ]:
import shap

DOMAIN = {
    "traffic_volume": "more vehicles on the road raise congestion",
    "incident_reports": "breakdowns and accidents block lanes and slow the whole corridor",
    "roadwork": "active construction reduces usable capacity",
    "public_transport_usage": "higher usage often tracks denser corridors; a rise can also mean fewer private vehicles",
    "traffic_signal_compliance": "poor compliance disrupts junction discharge and creates blockages",
    "parking_usage": "kerb parking reduces effective road width",
    "pedestrian_and_cyclist_count": "pedestrian-heavy streets are typically lower-vehicle streets, so the link can be negative",
    "rain_sum": "rain slows traffic and triggers waterlogging in Bengaluru's low-lying junctions",
    "rain_sum_next": "tomorrow's rain forecast", "temp_mean": "weather comfort / season proxy", "humidity_mean": "monsoon-season proxy",
    "wind_max": "storm / monsoon proxy", "cloud_mean": "monsoon-season proxy",
    "weekend": "weekend travel differs from the weekday office commute", "weekend_n": "tomorrow is a weekend day",
    "monsoon": "June-September monsoon", "monsoon_n": "tomorrow is in monsoon season",
    "cong_lag1": "yesterday's congestion", "cong_roll7": "last week's average congestion", "sat_share7": "how often the road saturated in the last week",
    "incident_rate": "incidents per 10k vehicles (risk normalised for exposure)", "pressure": "volume x capacity utilisation",
    "roadwork_x_vol": "roadwork matters more when volume is high", "rain_x_vol": "rain matters more when volume is high"}

ROAD_NAME = dict(zip(mf.ROAD_COLS, mf.ROADS))   # keep original spelling (e.g. "CMH Road")

def shap_values_for(model, bg, Xs):
    tree_types = (RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor, XGBRegressor, LGBMRegressor, CatBoostRegressor)
    if isinstance(model, tree_types):
        return np.asarray(shap.TreeExplainer(model).shap_values(Xs))
    return shap.Explainer(model.predict, shap.maskers.Independent(bg, max_samples=len(bg)))(Xs, silent=True).values   # Ridge pipelines, HistGradientBoosting

def explain_model(model, Xtr, Xte, yte, title, fname, shap_n=300, top=8):
    pi = permutation_importance(model, Xte, yte, n_repeats=10, random_state=SEED, scoring="r2", n_jobs=1)
    pimp = pd.DataFrame({"perm_importance": pi.importances_mean, "std": pi.importances_std}, index=Xte.columns).sort_values("perm_importance", ascending=False)
    bg, Xs = Xtr.sample(min(150, len(Xtr)), random_state=SEED), Xte.sample(min(shap_n, len(Xte)), random_state=SEED)
    sv = shap_values_for(model, bg, Xs)
    fig = plt.figure(figsize=(15, 5.5))
    plt.subplot(1, 2, 1); pimp.head(12)["perm_importance"][::-1].plot.barh(xerr=pimp.head(12)["std"][::-1]); plt.title("Permutation importance (drop in test R2)")
    plt.subplot(1, 2, 2); shap.summary_plot(sv, Xs, show=False, max_display=12, plot_size=None); plt.title("SHAP summary (red = high feature value)")
    plt.suptitle(title, fontweight="bold"); plt.tight_layout(); savefig(fname); plt.show()
    imp = pd.Series(np.abs(sv).mean(0), index=Xte.columns).sort_values(ascending=False)
    lines = []
    for f in imp.index[:top]:
        j = list(Xte.columns).index(f)
        if f.startswith("road_"):
            on = Xs[f].values == 1
            eff = sv[on, j].mean() if on.any() else 0.0
            lines.append(f"{ROAD_NAME.get(f, f[5:])}: being this road shifts the prediction by {eff:+.1f} points vs the average road")
            continue
        rho = sps.spearmanr(Xs[f], sv[:, j])[0]
        direction = "increases" if rho > 0.1 else "decreases" if rho < -0.1 else "has a mixed / weak effect on"
        lines.append(f"{f}: a higher value {direction} predicted congestion (mean |SHAP| {imp[f]:.2f})" + (f" — {DOMAIN[f]}" if f in DOMAIN else ""))
    return pimp, imp, lines

pimp_B, shap_B, lines_B = explain_model(final_B, Xb_train, Xb_test, yb_test, f"Task B — what drives estimated congestion ({final_B_name})", "xai_B")
print("What the what-if model learned:")
for l in lines_B: print(" •", l)

**Reality check — how much is traffic volume alone?** The permutation chart shows one feature dominating. Before trusting a high R², measure what that single feature (and the road) can do by itself:

In [ ]:
def _quick_r2(cols):
    m = HistGradientBoostingRegressor(max_iter=200, learning_rate=0.05, max_depth=4, random_state=SEED).fit(devb[cols], devb["congestion_level"])
    return r2_score(yb_test, np.clip(m.predict(testb[cols]), 0, 100))
vol_only_r2 = _quick_r2(["traffic_volume"]); vol_road_r2 = _quick_r2(["traffic_volume"] + mf.ROAD_COLS); all_r2 = mB["R2"]
print(f"test R2 — traffic_volume only: {vol_only_r2:.3f} | traffic_volume + road: {vol_road_r2:.3f} | all selected drivers ({final_B_name}): {all_r2:.3f}")
print("=> In this dataset the congestion index is almost a (saturating) function of traffic volume; the other drivers add little.")

**What this means.** The what-if estimator is accurate, but mostly because *congestion here is almost determined by traffic volume* — roughly a saturating curve that reaches the ceiling of 100 at high volume. Incidents, roadwork, weather, parking and public transport have small or undetectable extra effects in this data. That is a legitimate and useful finding (volume is the lever), and it is the honest way to describe the 0.97: **do not present it as a rich multi-factor model**.

In [ ]:
pimp_A, shap_A, lines_A = explain_model(final_A, X_train, X_test, y_test, f"Task A — what drives tomorrow's congestion ({final_A_name})", "xai_A")
print("What the forecast model learned:")
for l in lines_A: print(" •", l)

**Interpretation for Bengaluru.** Task B's top drivers tell you *what causes congestion in this data*; Task A's drivers (often the road identity itself) tell you *which corridors run persistently hot*. Both are consistent with how the city behaves: structural bottlenecks matter more than day-to-day variation, and volume, incidents and roadwork act through reduced effective capacity. The auto-generated bullets above state the direction measured from the data — the "why" text is a plausible mechanism, not a causal proof (this is observational data).

## Phase 15 — Save the models and build the prediction system
The saved models are **refit on all available data** (development + test) after evaluation, so the deployed model has seen the most recent period. The reported metrics are from the held-out evaluation above.

In [ ]:
import joblib
Xall_A = pd.concat([X_train, X_test]); yall_A = pd.concat([y_train, y_test])
Xall_B = pd.concat([Xb_train, Xb_test]); yall_B = pd.concat([yb_train, yb_test])
forecast_prod = clone(final_A).fit(Xall_A, yall_A)
whatif_prod = clone(final_B).fit(Xall_B, yall_B)
alert_prod = clone(alert_model).fit(pd.concat([Xa_train, Xa_test]), pd.concat([sat_train, sat_test]))
scaler_B = StandardScaler().fit(Xall_B[[c for c in FEATS_B if c in mf.OPS + mf.WX + ["roadwork"]]])   # reference scaler for the numeric drivers

INPUT_COLS = mf.OPS + ["roadwork"] + mf.WX + mf.WX_NEXT + mf.LAGS
road_defaults = df_fc.groupby("road")[INPUT_COLS].median().round(3)
road_area = tr.drop_duplicates("road").set_index("road")["area"].to_dict()
input_ranges = {c: [float(df_fc[c].quantile(.01)), float(df_fc[c].quantile(.99))] for c in INPUT_COLS}

bundle = dict(forecast=forecast_prod, alert=alert_prod, whatif=whatif_prod,
              feats_forecast=FEATS_A, feats_alert=FEATS_ALERT, feats_whatif=FEATS_B,
              names=dict(forecast=final_A_name, alert=alert_name, whatif=final_B_name))
joblib.dump(bundle, f"{PROJECT_DIR}/model.pkl")
joblib.dump(scaler_B, f"{PROJECT_DIR}/scaler.pkl")
joblib.dump(bundle, f"{PROJECT_DIR}/models/model_bundle.pkl")

meta = dict(project="Bengaluru Smart Mobility Intelligence", created=str(dt.date.today()),
            roads=mf.ROADS, road_area=road_area, input_cols=INPUT_COLS, input_ranges=input_ranges,
            road_defaults=road_defaults.to_dict(orient="index"),
            models=bundle["names"], feats=dict(forecast=FEATS_A, alert=FEATS_ALERT, whatif=FEATS_B),
            metrics=dict(forecast_test=mA, whatif_test=mB,
                         forecast_cv_r2=float(pick_A["cv_r2"]), whatif_cv_r2=float(pick_B["cv_r2"]),
                         per_road_mean_test_r2=float(rm["Test R2"]), whatif_volume_only_r2=float(vol_only_r2),
                         alert_test_auc=float(roc_auc_score(sat_test, p_alert)), alert_base_rate=float(sat_test.mean())),
            importance=dict(whatif=shap_B.head(15).round(4).to_dict(), forecast=shap_A.head(15).round(4).to_dict()),
            explanations=dict(whatif=lines_B, forecast=lines_A),
            comparison=dict(forecast=res_A.drop(columns=["Base"]).round(4).to_dict(orient="records"),
                            whatif=res_B.drop(columns=["Base"]).round(4).to_dict(orient="records")),
            train_period=[str(dev["target_date"].min().date()), str(dev["target_date"].max().date())],
            test_period=[str(test["target_date"].min().date()), str(test["target_date"].max().date())])
json.dump(meta, open(f"{PROJECT_DIR}/meta.json", "w"), indent=1, default=float)
print("saved:", sorted(os.listdir(PROJECT_DIR)))

In [ ]:
CONGESTION_BANDS = [(0, 50, "Light"), (50, 75, "Moderate"), (75, 90, "Heavy"), (90, 101, "Severe")]
def band(x):
    return next(n for lo, hi, n in CONGESTION_BANDS if lo <= x < hi)

def _row(road, date, overrides=None):
    d = road_defaults.loc[road].to_dict()
    d.update({k: v for k, v in (overrides or {}).items() if k in INPUT_COLS})
    if overrides and "rain_sum" in overrides and "rain_sum_next" not in overrides:
        d["rain_sum_next"] = overrides["rain_sum"]
    return pd.DataFrame([{**d, "road": road, "date": pd.Timestamp(date)}])

def predict_tomorrow(road, date, **inputs):
    # expected congestion tomorrow and the probability that it saturates; inputs override the road's typical values
    f = mf.build_features(_row(road, date, inputs))
    exp_c = float(np.clip(forecast_prod.predict(f[FEATS_A])[0], 0, 100)); p = float(alert_prod.predict_proba(f[FEATS_ALERT])[0, 1])
    typical = float(df_fc.loc[df_fc["road"] == road, "target"].mean())
    return dict(road=road, area=road_area[road], expected_congestion=round(exp_c, 1), band=band(exp_c),
                p_saturated=round(p, 3), road_typical=round(typical, 1),
                note=f"{road}: expected {exp_c:.0f}/100 ({band(exp_c)}); {p:.0%} chance of saturating (usual level {typical:.0f}).")

def estimate_congestion(road, date, **inputs):
    # what-if estimate of congestion given the drivers; inputs override the road's typical values
    f = mf.build_features(_row(road, date, inputs))
    c = float(np.clip(whatif_prod.predict(f[FEATS_B])[0], 0, 100))
    return dict(road=road, area=road_area[road], estimated_congestion=round(c, 1), band=band(c))

def sensitivity(road, date, **inputs):
    # change one driver at a time to see its effect on the what-if estimate
    base = estimate_congestion(road, date, **inputs)["estimated_congestion"]
    d = road_defaults.loc[road]; cur = {**d.to_dict(), **inputs}
    trials = {"Roadwork active": dict(roadwork=1.0), "Roadwork off": dict(roadwork=0.0),
              "+3 incidents": dict(incident_reports=cur["incident_reports"] + 3), "Traffic volume +20%": dict(traffic_volume=cur["traffic_volume"] * 1.2),
              "Traffic volume -20%": dict(traffic_volume=cur["traffic_volume"] * 0.8), "Heavy rain (30 mm)": dict(rain_sum=30.0),
              "Public transport +20%": dict(public_transport_usage=cur["public_transport_usage"] * 1.2)}
    out = {k: round(estimate_congestion(road, date, **{**inputs, **v})["estimated_congestion"] - base, 1) for k, v in trials.items()}
    return base, pd.Series(out, name="change in estimated congestion (points)")

road0 = mf.ROADS[0]
print(predict_tomorrow(road0, "2024-08-12")["note"])
# start from a calmer-than-usual day (30th percentile volume): on a day that already sits at the 100 ceiling no driver can push it higher
calm_vol = float(df_wi.loc[df_wi["road"] == road0, "traffic_volume"].quantile(0.30))
base_, sens_ = sensitivity(road0, "2024-08-12", traffic_volume=calm_vol)
print(f"What-if estimate for {road0} at {calm_vol:,.0f} vehicles: {base_:.0f}/100"); display(sens_.to_frame())

### Interactive prediction interface (works inside Colab)

In [ ]:
try:
    import ipywidgets as W
    from IPython.display import display as _disp, clear_output
    road_dd = W.Dropdown(options=mf.ROADS, description="Road")
    date_tx = W.Text(value="2024-08-12", description="Date")
    def _sl(col, label, step=1.0):
        lo, hi = input_ranges[col]; return W.FloatSlider(value=float(road_defaults.loc[road0, col]), min=lo, max=hi, step=step, description=label, style={"description_width": "150px"}, layout=W.Layout(width="520px"))
    sliders = {"traffic_volume": _sl("traffic_volume", "Traffic volume", 100.0), "incident_reports": _sl("incident_reports", "Incidents"),
               "public_transport_usage": _sl("public_transport_usage", "Public transport %"), "traffic_signal_compliance": _sl("traffic_signal_compliance", "Signal compliance %"),
               "parking_usage": _sl("parking_usage", "Parking usage %"), "pedestrian_and_cyclist_count": _sl("pedestrian_and_cyclist_count", "Pedestrians+cyclists"),
               "rain_sum": _sl("rain_sum", "Rain today (mm)", 0.5), "temp_mean": _sl("temp_mean", "Mean temp (C)", 0.5)}
    roadwork_cb = W.Checkbox(value=False, description="Roadwork active")
    out = W.Output(); btn = W.Button(description="Predict", button_style="primary")
    def _sync(_=None):
        for k, s in sliders.items(): s.value = float(road_defaults.loc[road_dd.value, k])
    road_dd.observe(_sync, names="value")
    def _run(_):
        with out:
            clear_output(); inp = {k: s.value for k, s in sliders.items()}; inp["roadwork"] = float(roadwork_cb.value)
            e = estimate_congestion(road_dd.value, date_tx.value, **inp); f = predict_tomorrow(road_dd.value, date_tx.value, **inp)
            print(f"WHAT-IF ESTIMATE (these conditions): {e['estimated_congestion']:.0f}/100 — {e['band']}")
            print("FORECAST FOR TOMORROW:", f["note"])
    btn.on_click(_run)
    _disp(W.VBox([road_dd, date_tx, *sliders.values(), roadwork_cb, btn, out]))
except ImportError:
    print("ipywidgets not available — use predict_tomorrow(...) / estimate_congestion(...) above, or the Streamlit app.")

## Phase 16 — Mobility intelligence layer
The models answer "how much?". This layer answers the planning questions: *which areas, which conditions, when, why, and where to act*. Effects of weather, roadwork and incidents are measured **after removing each road's own average**, otherwise a road that happens to have more roadwork days would be mistaken for a roadwork effect.

In [ ]:
obs = tr.copy(); obs["date"] = pd.to_datetime(obs["date"])
obs["dow"] = obs["date"].dt.day_name(); obs["month"] = obs["date"].dt.month; obs["year"] = obs["date"].dt.year
obs["saturated"] = (obs["congestion_level"] >= 99.9).astype(int)
obs["cong_vs_road"] = obs["congestion_level"] - obs.groupby("road")["congestion_level"].transform("mean")   # congestion relative to the road's own usual level
obs["season"] = obs["month"].map(lambda m: "Winter (Dec-Feb)" if m in (12, 1, 2) else "Summer (Mar-May)" if m in (3, 4, 5) else "Monsoon (Jun-Sep)" if m in (6, 7, 8, 9) else "Post-monsoon (Oct-Nov)")

def mm(s): return (s - s.min()) / (s.max() - s.min() + 1e-9)
road_stats = obs.groupby(["area", "road"]).agg(
    volume=("traffic_volume", "mean"), congestion=("congestion_level", "mean"), saturated_share=("saturated", "mean"),
    capacity_util=("road_capacity_utilization", "mean"), travel_time_index=("travel_time_index", "mean"), speed=("average_speed", "mean"),
    incidents=("incident_reports", "mean"), compliance=("traffic_signal_compliance", "mean"), public_transport=("public_transport_usage", "mean"),
    days=("date", "size")).reset_index()
road_stats["pressure_index"] = 100 * (0.3 * mm(road_stats["volume"]) + 0.3 * mm(road_stats["capacity_util"]) + 0.2 * mm(road_stats["travel_time_index"]) + 0.2 * mm(road_stats["congestion"]))
area_stats = obs.groupby("area").agg(volume=("traffic_volume", "mean"), congestion=("congestion_level", "mean"), saturated_share=("saturated", "mean"),
                                     capacity_util=("road_capacity_utilization", "mean"), speed=("average_speed", "mean")).reset_index()
area_stats["pressure_index"] = 100 * (0.4 * mm(area_stats["volume"]) + 0.3 * mm(area_stats["capacity_util"]) + 0.3 * mm(area_stats["congestion"]))
area_stats = area_stats.sort_values("pressure_index", ascending=False)

fig, ax = plt.subplots(1, 2, figsize=(15, 5))
sns.barplot(data=area_stats, y="area", x="pressure_index", ax=ax[0], color="#4C72B0"); ax[0].set_title("Traffic Pressure Index by area (0-100)")
rs = road_stats.sort_values("pressure_index", ascending=False)
sns.barplot(data=rs, y="road", x="pressure_index", ax=ax[1], color="#C44E52"); ax[1].set_title("Traffic Pressure Index by road")
plt.tight_layout(); savefig("pressure_index"); plt.show()
display(road_stats.sort_values("pressure_index", ascending=False).head(6).round(2))

**Q: Which areas experience higher traffic pressure?** The Pressure Index blends volume, capacity utilisation, travel-time index and congestion (min-max scaled, weights shown in the code). Because of the min-max scaling it is a *relative* index: the busiest area/road scores 100 and the quietest scores 0 by construction (a 0 does not mean "no traffic"). The weights are a transparent policy choice, not a fitted quantity.

In [ ]:
# ---- How does traffic vary across time? ----
order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
fig, ax = plt.subplots(1, 3, figsize=(17, 4))
d1 = obs.groupby("dow")["congestion_level"].mean().reindex(order); ax[0].bar(range(7), d1.values, color="#4C72B0"); ax[0].set_xticks(range(7)); ax[0].set_xticklabels([o[:3] for o in order]); ax[0].set_title("Mean congestion by weekday"); ax[0].set_ylim(d1.min() - 5, d1.max() + 3)
m1 = obs.groupby("month")["congestion_level"].mean(); ax[1].plot(m1.index, m1.values, "o-"); ax[1].set_title("Mean congestion by month"); ax[1].set_xlabel("month")
s1 = obs.groupby("season")["congestion_level"].mean(); ax[2].bar(range(len(s1)), s1.values, color="#55A868"); ax[2].set_xticks(range(len(s1))); ax[2].set_xticklabels([x.split(" (")[0] for x in s1.index]); ax[2].set_title("Mean congestion by season"); ax[2].set_ylim(s1.min() - 5, s1.max() + 3)
plt.tight_layout(); savefig("temporal_patterns"); plt.show()
p_dow = sps.kruskal(*[g["cong_vs_road"].values for _, g in obs.groupby("dow")])[1]
p_mon = sps.kruskal(*[g["cong_vs_road"].values for _, g in obs.groupby("month")])[1]
p_sea = sps.kruskal(*[g["cong_vs_road"].values for _, g in obs.groupby("season")])[1]
print(f"Kruskal-Wallis on congestion relative to road average -> weekday p={p_dow:.3f} | month p={p_mon:.3f} | season p={p_sea:.3f}")
print(f"weekday range: {d1.max() - d1.min():.1f} pts | month range: {m1.max() - m1.min():.1f} pts | season range: {s1.max() - s1.min():.1f} pts")

In [ ]:
# ---- Capacity utilisation vs congestion; incidents; public transport ----
fig, ax = plt.subplots(1, 3, figsize=(17, 4.2))
# capacity utilisation has many tied values, so quantile bins collapse; equal-width bins show the real shape
b = pd.cut(obs["road_capacity_utilization"], 12); s = obs.groupby(b, observed=True)["congestion_level"].agg(["mean", "sem", "size"]); s = s[s["size"] >= 20]
ax[0].errorbar([iv.mid for iv in s.index], s["mean"], 1.96 * s["sem"], marker="o"); ax[0].axhline(90, color="r", ls="--", lw=.8); ax[0].set_title("Congestion vs capacity utilisation (12 equal-width bins)"); ax[0].set_xlabel("capacity utilisation (%)"); ax[0].set_ylabel("mean congestion")
ib = obs["incident_reports"].clip(upper=3).astype(int).astype(str).replace({"3": "3+"}); s2 = obs.groupby(ib)["cong_vs_road"].agg(["mean", "sem", "size"])
ax[1].bar(s2.index, s2["mean"], yerr=1.96 * s2["sem"], color="#DD8452"); ax[1].axhline(0, color="k", lw=.8); ax[1].set_title("Congestion vs road's own average, by incidents that day"); ax[1].set_xlabel("incident reports")
pb = pd.qcut(obs["public_transport_usage"], 10, duplicates="drop"); s3 = obs.groupby(pb)["cong_vs_road"].agg(["mean", "sem"]).reset_index(drop=True)
ax[2].errorbar(range(len(s3)), s3["mean"], 1.96 * s3["sem"], marker="o", color="#55A868"); ax[2].axhline(0, color="k", lw=.8); ax[2].set_title("Congestion vs road average by public-transport usage decile")
plt.tight_layout(); savefig("capacity_incident_pt"); plt.show()
rho_cap = sps.spearmanr(obs["road_capacity_utilization"], obs["congestion_level"])[0]
rho_inc = sps.spearmanr(obs["incident_reports"], obs["cong_vs_road"])[0]
rho_pt = sps.spearmanr(obs["public_transport_usage"], obs["cong_vs_road"])[0]
rho_pt_raw = sps.spearmanr(obs["public_transport_usage"], obs["congestion_level"])[0]
print(f"capacity util vs congestion Spearman = {rho_cap:+.2f} | incidents vs congestion (within road) = {rho_inc:+.2f} | public transport vs congestion: raw {rho_pt_raw:+.2f}, within road {rho_pt:+.2f}")

In [ ]:
# ---- weather and roadwork ----
def mean_diff_ci(a, b, n=500):
    rng = np.random.default_rng(SEED); a, b = np.asarray(a), np.asarray(b)
    diffs = [rng.choice(a, len(a)).mean() - rng.choice(b, len(b)).mean() for _ in range(n)]
    return a.mean() - b.mean(), np.percentile(diffs, [2.5, 97.5])

rw = obs.groupby(obs["roadwork"].map({1.0: "Roadwork", 0.0: "No roadwork"}))["cong_vs_road"]
d_rw, ci_rw = mean_diff_ci(obs.loc[obs["roadwork"] == 1, "cong_vs_road"], obs.loc[obs["roadwork"] == 0, "cong_vs_road"])
p_rw = sps.mannwhitneyu(obs.loc[obs["roadwork"] == 1, "cong_vs_road"], obs.loc[obs["roadwork"] == 0, "cong_vs_road"])[1]

obs = obs.merge(weather_daily[["date", "rain_sum"]], on="date", how="left")
obs["rain_bin"] = pd.cut(obs["rain_sum"], [-0.1, 0.1, 5, 20, 500], labels=["dry", "light (<5mm)", "moderate (5-20)", "heavy (>20)"])
wc = obs.groupby("weather_conditions")["cong_vs_road"].agg(["mean", "sem", "size"])
rb = obs.groupby("rain_bin")["cong_vs_road"].agg(["mean", "sem", "size"])
p_wc = sps.kruskal(*[g["cong_vs_road"].values for _, g in obs.groupby("weather_conditions")])[1]
p_rain = sps.kruskal(*[g["cong_vs_road"].values for _, g in obs.dropna(subset=["rain_bin"]).groupby("rain_bin")])[1]

fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].bar(["No roadwork", "Roadwork"], [obs.loc[obs["roadwork"] == 0, "cong_vs_road"].mean(), obs.loc[obs["roadwork"] == 1, "cong_vs_road"].mean()], color=["#4C72B0", "#C44E52"]); ax[0].axhline(0, color="k", lw=.8); ax[0].set_title(f"Roadwork effect: {d_rw:+.1f} pts (95% CI {ci_rw[0]:+.1f} to {ci_rw[1]:+.1f})", fontsize=10)
ax[1].bar(wc.index, wc["mean"], yerr=1.96 * wc["sem"], color="#8172B2"); ax[1].axhline(0, color="k", lw=.8); ax[1].set_title(f"Weather condition label (Kruskal p={p_wc:.2f})", fontsize=10); ax[1].tick_params(axis="x", rotation=25)
ax[2].bar(rb.index.astype(str), rb["mean"], yerr=1.96 * rb["sem"], color="#64B5CD"); ax[2].axhline(0, color="k", lw=.8); ax[2].set_title(f"Measured daily rainfall (Kruskal p={p_rain:.2f})", fontsize=10); ax[2].tick_params(axis="x", rotation=25)
for a in ax: a.set_ylabel("congestion vs road's own average")
plt.tight_layout(); savefig("weather_roadwork"); plt.show()
print(f"roadwork: {d_rw:+.2f} pts, Mann-Whitney p={p_rw:.3f} | weather label p={p_wc:.3f} | measured rain p={p_rain:.3f}")

**Reading the effects.** A confidence interval that spans 0 and a p-value above 0.05 mean *no detectable effect in this data* — which is a result, not a failure. We state effects only when they are detectable (see the auto-generated insight list at the end of this phase).

In [ ]:
# ---- Which locations require attention? composite attention score ----
hot = road_stats.copy()
hot["incident_rate"] = hot["incidents"] / (hot["volume"] / 1e4)
hot["compliance_gap"] = 100 - hot["compliance"]
hot["attention_score"] = 100 * (0.35 * mm(hot["congestion"]) + 0.25 * mm(hot["saturated_share"]) + 0.20 * mm(hot["incident_rate"]) + 0.20 * mm(hot["compliance_gap"]))
hot = hot.sort_values("attention_score", ascending=False).reset_index(drop=True)
hot["priority"] = pd.cut(hot["attention_score"], [-1, 33, 66, 101], labels=["Monitor", "Elevated", "Act now"])
display(hot[["area", "road", "congestion", "saturated_share", "incident_rate", "compliance", "attention_score", "priority"]].head(8).round(2))
plt.figure(figsize=(8, 5)); sns.barplot(data=hot, y="road", x="attention_score", hue="priority", dodge=False, palette={"Monitor": "#55A868", "Elevated": "#DD8452", "Act now": "#C44E52"})
plt.title("Locations requiring attention (congestion, saturation, incident rate, signal non-compliance)"); plt.tight_layout(); savefig("hotspots"); plt.show()

### Risk / impact model — expected incidents
A Poisson gradient-boosting model of daily `incident_reports` from congestion, weather, roadwork, calendar and the road (chronological split). It turns a forecast into an *expected incident count* and, combined with the fatal share from the crash data, into a **fatal-exposure index** — a ranking aid, not a casualty forecast.

In [ ]:
INC_FEATS = ["congestion_level", "traffic_volume", "rain_sum", "temp_mean", "roadwork", "dow_sin", "dow_cos", "weekend"] + mf.ROAD_COLS
incd = df_wi.dropna(subset=INC_FEATS + ["incident_reports"]).sort_values("date")
cut_i = incd["date"].iloc[int(0.85 * len(incd))]
i_tr, i_te = incd[incd["date"] < cut_i], incd[incd["date"] >= cut_i]
inc_model = HistGradientBoostingRegressor(loss="poisson", max_iter=250, learning_rate=0.05, max_depth=3, l2_regularization=1.0, random_state=SEED).fit(i_tr[INC_FEATS], i_tr["incident_reports"])
yp, yt = inc_model.predict(i_te[INC_FEATS]), i_te["incident_reports"].values
base_mean = np.full_like(yp, i_tr["incident_reports"].mean())
print(f"Poisson deviance: model {mean_poisson_deviance(yt, yp):.3f} vs train-mean baseline {mean_poisson_deviance(yt, base_mean):.3f} | MAE {mean_absolute_error(yt, yp):.3f} vs {mean_absolute_error(yt, base_mean):.3f}")
pi_inc = permutation_importance(inc_model, i_te[INC_FEATS], yt, n_repeats=5, random_state=SEED, scoring="neg_mean_poisson_deviance")
pi_inc = pd.Series(pi_inc.importances_mean, index=INC_FEATS).sort_values(ascending=False)
print("Top drivers of incidents:"); print(pi_inc.head(5).round(4).to_string())
INC_P95 = float(np.quantile(inc_model.predict(i_tr[INC_FEATS]), 0.95))

## Road safety (Road Crashes 2007–2025)

In [ ]:
cr = crash[["Year", "Fatal Road Accidents", "Killed", "Non-Fatal Road Accidents", "Total"]].copy().sort_values("Year").reset_index(drop=True)
cr["killed_per_100_crashes"] = 100 * cr["Killed"] / cr["Total"]
cr["fatal_share_pct"] = 100 * cr["Fatal Road Accidents"] / cr["Total"]
cr["killed_per_fatal_crash"] = cr["Killed"] / cr["Fatal Road Accidents"]

def yearly_forecast(df, ycol, target, holdout=3, covid_years=(2020, 2021)):
    d = df[[ycol, target]].dropna().sort_values(ycol).reset_index(drop=True); d["covid"] = d[ycol].isin(covid_years).astype(int)
    f = [ycol, "covid"]; a, b = d.iloc[:-holdout], d.iloc[-holdout:]
    preds = {"linear_trend": LinearRegression().fit(a[f], a[target]).predict(b[f]), "naive_last": np.repeat(a[target].iloc[-1], holdout), "mean_last3": np.repeat(a[target].iloc[-3:].mean(), holdout)}
    mae = {k: float(mean_absolute_error(b[target], v)) for k, v in preds.items()}; best = min(mae, key=mae.get); ny = int(d[ycol].max()) + 1
    val = (float(LinearRegression().fit(d[f], d[target]).predict(pd.DataFrame({ycol: [ny], "covid": [0]}))[0]) if best == "linear_trend"
           else float(d[target].iloc[-1]) if best == "naive_last" else float(d[target].iloc[-3:].mean()))
    return dict(target=target, method=best, backtest_mae=mae, next_year=ny, forecast=val)

crash_fc = {t: yearly_forecast(cr, "Year", t) for t in ["Total", "Killed", "fatal_share_pct", "killed_per_100_crashes"]}
for t, r in crash_fc.items():
    print(f"{t:24s} best back-test method: {r['method']:13s} -> {r['next_year']}: {r['forecast']:.1f}")

fig = make_subplots(rows=1, cols=3, subplot_titles=("Crashes per year", "People killed per year", "Fatality rate (killed per 100 crashes)"))
fig.add_trace(go.Scatter(x=cr["Year"], y=cr["Total"], mode="lines+markers", name="Total crashes"), 1, 1)
fig.add_trace(go.Scatter(x=cr["Year"], y=cr["Killed"], mode="lines+markers", name="Killed", line=dict(color="#C44E52")), 1, 2)
fig.add_trace(go.Scatter(x=cr["Year"], y=cr["killed_per_100_crashes"], mode="lines+markers", name="Killed / 100 crashes", line=dict(color="#8172B2")), 1, 3)
fig.update_layout(height=340, showlegend=False, title_text="Road safety trends"); fig.show()
CITY_FATAL_SHARE = float(cr["Fatal Road Accidents"].iloc[-3:].sum() / cr["Total"].iloc[-3:].sum())
print(f"Fatal share of crashes (last 3 years): {CITY_FATAL_SHARE:.1%} | check whether the final year in the file is a full year before trusting trend forecasts")

## Traffic enforcement (BTP cases and fines)

In [ ]:
bt = btp.sort_values("YEAR").reset_index(drop=True).copy()
bt["avg_fine_per_case"] = bt["Total Fine Collected"] / bt["Total Cases Booked"]
bt["automation_share_pct"] = 100 * bt["Automation cases"] / bt["Total Cases Booked"]
bt["towing_share_pct"] = 100 * bt["Towing Cases"] / bt["Total Cases Booked"]
fc_cases = yearly_forecast(bt, "YEAR", "Total Cases Booked"); fc_fine = yearly_forecast(bt, "YEAR", "Total Fine Collected")
print(f"Forecast {fc_cases['next_year']}: {fc_cases['forecast']:,.0f} cases ({fc_cases['method']}), fines Rs {fc_fine['forecast'] / 1e7:,.1f} crore ({fc_fine['method']})")

j = bt.merge(cr.rename(columns={"Year": "YEAR"}), on="YEAR")
ec = ["Total Cases Booked", "automation_share_pct", "avg_fine_per_case"]; cc = ["Total", "Killed", "killed_per_100_crashes"]
same = j[ec + cc].corr(method="spearman").loc[ec, cc]
lead = pd.concat([j[ec], j[cc].shift(-1)], axis=1).corr(method="spearman").loc[ec, cc]
print(f"Overlapping years: {len(j)}  (descriptive only — n is tiny and this is not causal evidence)")
print("Spearman, same year:"); display(same.round(2)); print("Enforcement(t) vs crashes(t+1):"); display(lead.round(2))

fig = make_subplots(rows=1, cols=3, subplot_titles=("Cases booked (millions)", "Automated share of cases (%)", "Total fines collected (Rs crore)"))
fig.add_trace(go.Bar(x=bt["YEAR"], y=bt["Total Cases Booked"] / 1e6, marker_color="#4C72B0"), 1, 1)
fig.add_trace(go.Scatter(x=bt["YEAR"], y=bt["automation_share_pct"], mode="lines+markers", line=dict(color="#55A868")), 1, 2)
fig.add_trace(go.Bar(x=bt["YEAR"], y=bt["Total Fine Collected"] / 1e7, marker_color="#DD8452"), 1, 3)
fig.update_layout(height=340, showlegend=False, title_text="Enforcement trends"); fig.show()

enforcement_info = dict(forecast_year=fc_cases["next_year"], cases_forecast=fc_cases["forecast"], cases_per_day=fc_cases["forecast"] / 365.0,
                        avg_fine=float(bt["avg_fine_per_case"].iloc[-1]), automation_share=float(bt["automation_share_pct"].iloc[-1] / 100))

### Auto-generated insight list (only claims the data supports)

In [ ]:
def sig(p): return "statistically detectable" if p < 0.05 else "not detectable"
top_area, top_road = area_stats.iloc[0], rs.iloc[0]
INSIGHTS = [
    f"Highest traffic pressure: {top_area['area']} (index {top_area['pressure_index']:.0f}/100) among areas; {top_road['road']} ({top_road['area']}) among roads (index {top_road['pressure_index']:.0f}).",
    f"{obs['saturated'].mean():.0%} of road-days sit at the congestion ceiling of 100; saturation ranges from {road_stats['saturated_share'].min():.0%} to {road_stats['saturated_share'].max():.0%} across roads — WHICH road it is explains {eta2:.0%} of next-day congestion variance.",
    f"Time patterns: weekday differences are {sig(p_dow)} (p={p_dow:.2f}, range {d1.max() - d1.min():.1f} pts); monthly {sig(p_mon)} (p={p_mon:.2f}); seasonal {sig(p_sea)} (p={p_sea:.2f}).",
    f"Capacity utilisation and congestion move together (Spearman {rho_cap:+.2f}): congested roads are the ones that are already full.",
    f"Incidents: within-road correlation with congestion {rho_inc:+.2f}.",
    f"Roadwork effect on congestion vs the road's own average: {d_rw:+.1f} pts (95% CI {ci_rw[0]:+.1f} to {ci_rw[1]:+.1f}), {sig(p_rw)}.",
    f"Weather: label effect {sig(p_wc)} (p={p_wc:.2f}); measured rainfall effect {sig(p_rain)} (p={p_rain:.2f}).",
    f"Public transport usage vs congestion: raw Spearman {rho_pt_raw:+.2f}, within-road {rho_pt:+.2f}.",
    f"Locations needing most attention: {', '.join(hot['road'].head(3))}.",
    f"Road safety: {int(cr['Killed'].iloc[-1]):,} killed in {int(cr['Year'].iloc[-1])} vs peak {int(cr['Killed'].max()):,} in {int(cr.loc[cr['Killed'].idxmax(), 'Year'])}; fatality rate {cr['killed_per_100_crashes'].iloc[-1]:.1f} killed per 100 crashes.",
    f"Enforcement: {bt['Total Cases Booked'].iloc[-1] / 1e6:.2f} million cases in {int(bt['YEAR'].iloc[-1])}, of which {bt['automation_share_pct'].iloc[-1]:.0f}% automated; forecast {fc_cases['forecast'] / 1e6:.2f} million next year."]
for i in INSIGHTS: print("•", i)

## Phase 17 — Fusion: the final intelligence report
One function combines everything: forecast, alert probability, expected incidents, fatal-exposure index, enforcement context and — when a camera frame is supplied (Phase 18) — the **visual congestion index** from YOLO. The logic lives in a small shared module, `mobility_report.py`, so the notebook and the Streamlit app produce identical reports. The priority weights are a policy heuristic — expose them to the people who will use the tool.

In [ ]:
%%writefile mobility_report.py
# Shared prediction + intelligence-report logic (used by the notebook AND the Streamlit app)
import numpy as np, pandas as pd
import mobility_features as mf

BANDS = [(0, 50, "Light"), (50, 75, "Moderate"), (75, 90, "Heavy"), (90, 101, "Severe")]
PRIORITY_W = (0.45, 0.35, 0.20)     # congestion, incident risk, signal non-compliance

def band(x):
    return next(n for lo, hi, n in BANDS if lo <= x < hi)

def make_row(meta, road, date, ov=None):
    ov = ov or {}
    d = dict(meta["road_defaults"][road])
    d.update({k: v for k, v in ov.items() if k in meta["input_cols"]})
    if "rain_sum" in ov and "rain_sum_next" not in ov:
        d["rain_sum_next"] = ov["rain_sum"]
    return pd.DataFrame([{**d, "road": road, "date": pd.Timestamp(date)}])

def estimate_now(bundle, meta, road, date, ov=None):
    f = mf.build_features(make_row(meta, road, date, ov))
    return float(np.clip(bundle["whatif"].predict(f[bundle["feats_whatif"]])[0], 0, 100))

def forecast_tomorrow(bundle, meta, road, date, ov=None):
    f = mf.build_features(make_row(meta, road, date, ov))
    return (float(np.clip(bundle["forecast"].predict(f[bundle["feats_forecast"]])[0], 0, 100)),
            float(bundle["alert"].predict_proba(f[bundle["feats_alert"]])[0, 1]))

def intelligence_report(bundle, meta, road, date, ov=None, vision=None, vision_alpha=0.3):
    # vision: dict with visual_index (0-100), pcu, n_vehicles, counts  (from vision_utils.detect_frame / aggregated frames)
    ov = ov or {}
    exp_c, p_sat = forecast_tomorrow(bundle, meta, road, date, ov)
    nxt = pd.Timestamp(date) + pd.Timedelta(days=1)
    d = meta["road_defaults"][road]
    tmr = make_row(meta, road, nxt, {**ov, "congestion_level": exp_c, "rain_sum": ov.get("rain_sum_next", d["rain_sum_next"]), "temp_mean": d["temp_mean_next"]})
    exp_inc = float(bundle["incident"].predict(mf.build_features(tmr)[bundle["feats_incident"]])[0])
    p95 = bundle["inc_p95"]
    compliance = float(ov.get("traffic_signal_compliance", d["traffic_signal_compliance"]))
    cong_used = exp_c
    if vision is not None:
        cong_used = (1 - vision_alpha) * exp_c + vision_alpha * vision["visual_index"]
    w1, w2, w3 = PRIORITY_W
    priority = 100 * (w1 * cong_used / 100 + w2 * min(1.0, exp_inc / p95 if p95 > 0 else 0.0) + w3 * float(np.clip(1 - compliance / 100, 0, 1)))
    tier = "HIGH" if priority >= 65 else "MEDIUM" if priority >= 45 else "LOW"
    recs = []
    if cong_used >= 90 or p_sat >= 0.65:
        recs.append("Deploy peak-hour marshals / review signal timing on this corridor")
    if vision is not None and vision["visual_index"] >= 85:
        recs.append(f"Camera shows a heavy vehicle load right now (visual index {vision['visual_index']:.0f}/100): consider immediate diversion or marshal deployment")
    if exp_inc >= p95:
        recs.append("Pre-position a quick-response patrol (incident risk in the top 5% of history)")
    if compliance < 75:
        recs.append(f"Focus signal-violation enforcement (compliance {compliance:.0f}%)")
    if float(ov.get("rain_sum_next", d["rain_sum_next"])) >= 10:
        recs.append("Heavy rain forecast: waterlogging and low-visibility cautions")
    if not recs:
        recs.append("Routine monitoring")
    enf = meta["enforcement"]
    return dict(road=road, area=meta["road_area"][road], target_date=nxt.date().isoformat(), expected_congestion=exp_c, band=band(exp_c),
                p_saturated=p_sat, expected_incidents=exp_inc, fatal_exposure_index=exp_inc * meta["safety"]["fatal_share"],
                priority_score=priority, priority_tier=tier, recommendations=recs, congestion_used=cong_used,
                btp_context=dict(cases_per_day=enf["cases_per_day"], automation_share=enf["automation_share"]),
                vision=None if vision is None else {k: vision[k] for k in ("counts", "n_vehicles", "pcu", "visual_index")})

def format_report(r):
    s = [f"{r['road']} ({r['area']}) — outlook for {r['target_date']}",
         f"  Congestion forecast : {r['expected_congestion']:.0f}/100 [{r['band']}]   P(saturated) = {r['p_saturated']:.0%}",
         f"  Expected incidents  : {r['expected_incidents']:.2f}   fatal-exposure index {r['fatal_exposure_index']:.3f}",
         f"  Enforcement priority: {r['priority_score']:.0f}/100 -> {r['priority_tier']}"]
    if r["vision"]:
        s.append(f"  Camera frame        : {r['vision']['n_vehicles']:.0f} vehicles, {r['vision']['pcu']:.1f} PCU, visual index {r['vision']['visual_index']:.0f}/100")
    s.append("  Actions: " + "; ".join(r["recommendations"]))
    s.append(f"  BTP context         : ~{r['btp_context']['cases_per_day']:,.0f} cases/day citywide, {r['btp_context']['automation_share']:.0%} automated")
    return "\n".join(s)

In [ ]:
import importlib
import mobility_report as mr
importlib.reload(mr)

# add the incident-risk model and the safety / enforcement context to the saved bundle so the app can produce the same report
bundle.update(incident=inc_model, feats_incident=INC_FEATS, inc_p95=INC_P95)
meta["safety"] = dict(fatal_share=CITY_FATAL_SHARE, forecast={k: v["forecast"] for k, v in crash_fc.items()})
meta["enforcement"] = enforcement_info
meta["insights"] = INSIGHTS
joblib.dump(bundle, f"{PROJECT_DIR}/model.pkl"); joblib.dump(bundle, f"{PROJECT_DIR}/models/model_bundle.pkl")
json.dump(meta, open(f"{PROJECT_DIR}/meta.json", "w"), indent=1, default=float)

def mobility_intelligence_report(road, date, image=None, vision_fn=None, vision_alpha=0.3, **inputs):
    vision = vision_fn(image) if (image is not None and vision_fn is not None) else None
    return mr.intelligence_report(bundle, meta, road, date, inputs, vision, vision_alpha)
format_report = mr.format_report

ranking = pd.DataFrame([mobility_intelligence_report(r, "2024-08-12") for r in mf.ROADS]).sort_values("priority_score", ascending=False).reset_index(drop=True)
display(ranking[["road", "area", "expected_congestion", "p_saturated", "expected_incidents", "priority_score", "priority_tier"]].head(8).round(2))
print(format_report(mobility_intelligence_report(ranking.loc[0, "road"], "2024-08-12")))

## Phase 18 — YOLO CCTV vehicle detection (BMD-45)
A camera frame becomes a **congestion signal**: YOLO detects and classifies vehicles, each detection is weighted by its passenger-car-unit (PCU) factor (a bus loads the road more than a scooter), and the frame's PCU load is converted to a **visual congestion index (0–100)**. That index is fused into the intelligence report, and the detector is served in the Streamlit app's **CCTV tab** (image upload, video clip, live stream URL).

* **Data:** BMD-45 — 14 vehicle classes, CCTV frames from Bengaluru. Boxes are `[x, y, width, height]` in pixels and include the rider. We train on the official `train` split and validate on `validation`; the test-split labels are withheld by the authors, so a separate block of validation images is held out for the final check.
* **Hardware:** on a **T4 GPU** runtime the full configuration takes roughly 20–40 minutes. On a CPU runtime the notebook automatically switches to a reduced "lite" configuration (fewer images and epochs, lower accuracy) so the whole project still completes — switch to a GPU for a production-quality detector.
* **Honest scope:** BMD-45 cameras are not geo-linked to the roads in the traffic table, so the app lets the user pair a camera with a road; the visual index describes a single view, not the whole corridor.

In [ ]:
try:
    import torch
    HAS_GPU = torch.cuda.is_available()
except Exception:                      # torch missing or unusable -> CPU configuration
    torch, HAS_GPU = None, False
if HAS_GPU:
    TRAIN_N, VAL_N, HELD_N, YOLO_EPOCHS, YOLO_IMGSZ, YOLO_BATCH = 3000, 500, 200, 30, 640, 16
else:
    TRAIN_N, VAL_N, HELD_N, YOLO_EPOCHS, YOLO_IMGSZ, YOLO_BATCH = 600, 150, 100, 5, 480, 16
    print("No GPU detected -> LITE configuration (fewer images/epochs). For the full model: Runtime > Change runtime type > T4 GPU, then re-run from this cell.")
YOLO_DEVICE = 0 if HAS_GPU else "cpu"
YOLO_DIR = Path(ROOT) / "bmd45_yolo"
print(f"GPU: {HAS_GPU} | train {TRAIN_N} / val {VAL_N} / held-out {HELD_N} images | {YOLO_EPOCHS} epochs @ {YOLO_IMGSZ}px")

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics", "datasets", "opencv-python-headless"], check=True)

In [ ]:
%%writefile vision_utils.py
# YOLO vehicle detection -> PCU load -> visual congestion index (shared by the notebook AND the Streamlit app)
import json
from collections import Counter
import numpy as np

CLASS_NAMES = ["Hatchback", "Sedan", "SUV", "MUV", "Bus", "Truck", "Three-wheeler", "Two-wheeler", "LCV", "Mini-bus",
               "Tempo-traveller", "Bicycle", "Van", "Other"]
# approximate IRC-style urban passenger-car-unit factors; tune to policy
PCU = {"Hatchback": 1.0, "Sedan": 1.0, "SUV": 1.0, "MUV": 1.0, "Van": 1.0, "Other": 1.0, "Bus": 2.2, "Truck": 2.2, "Mini-bus": 2.2,
       "LCV": 1.4, "Tempo-traveller": 1.4, "Three-wheeler": 1.2, "Two-wheeler": 0.5, "Bicycle": 0.4}
PCU_BY_ID = np.array([PCU[c] for c in CLASS_NAMES], dtype=float)

def load_detector(weights):
    from ultralytics import YOLO
    return YOLO(weights)

def load_reference(path):
    return np.sort(np.array(json.load(open(path)), dtype=float))

def visual_index(pcu, ref):
    # percentile (0-100) of a frame's PCU load within the reference distribution of annotated frames
    return float(100.0 * np.searchsorted(ref, pcu, side="right") / len(ref))

def detect_frame(model, image, ref, conf=0.25, iou=0.5, imgsz=640):
    # image: file path | PIL image | numpy array in RGB order
    if isinstance(image, np.ndarray):
        image = np.ascontiguousarray(image[..., ::-1])            # ultralytics expects BGR for arrays
    r = model.predict(image, conf=conf, iou=iou, imgsz=imgsz, verbose=False)[0]
    cls = r.boxes.cls.cpu().numpy().astype(int)
    pcu = float(PCU_BY_ID[cls].sum()) if len(cls) else 0.0
    counts = {CLASS_NAMES[k]: int(v) for k, v in sorted(Counter(cls.tolist()).items())}
    return dict(counts=counts, n_vehicles=int(len(cls)), pcu=pcu, visual_index=visual_index(pcu, ref),
                annotated=np.ascontiguousarray(r.plot()[..., ::-1]))      # RGB for display

def aggregate(frames):
    # summarise several per-frame detections into one 'vision' dict for the intelligence report
    total = Counter()
    for f in frames:
        total.update(f["counts"])
    return dict(counts=dict(total), n_vehicles=float(np.mean([f["n_vehicles"] for f in frames])),
                pcu=float(np.mean([f["pcu"] for f in frames])), visual_index=float(np.mean([f["visual_index"] for f in frames])))

def video_frames(path, every_s=2.0, max_frames=24):
    # sample frames from a video file every `every_s` seconds -> list of (timestamp_s, RGB ndarray)
    import cv2
    cap = cv2.VideoCapture(path)
    if not cap.isOpened():
        raise IOError("could not open video")
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    step = max(1, int(round(fps * every_s)))
    out, i = [], 0
    while len(out) < max_frames:
        ok, frame = cap.read()
        if not ok:
            break
        if i % step == 0:
            out.append((i / fps, frame[..., ::-1].copy()))
        i += 1
    cap.release()
    return out

def grab_stream_frame(url, warmup=3):
    # grab one RGB frame from a live stream (RTSP / HTTP MJPEG / HLS) or webcam index
    import cv2
    cap = cv2.VideoCapture(int(url) if str(url).isdigit() else url)
    if not cap.isOpened():
        raise IOError("could not open the stream URL")
    frame = None
    for _ in range(max(1, warmup)):
        ok, f = cap.read()
        if ok:
            frame = f
    cap.release()
    if frame is None:
        raise IOError("stream opened but no frame was received")
    return frame[..., ::-1].copy()

def write_clip(images, out_path, fps=1.0, size=(960, 540)):
    # build a short demo clip from still frames (PIL images)
    import cv2
    vw = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, size)
    for im in images:
        vw.write(np.asarray(im.convert("RGB").resize(size))[..., ::-1])
    vw.release()

In [ ]:
from collections import Counter
from PIL import Image
from datasets import load_dataset
import vision_utils as vu
CLASS_NAMES, PCU_BY_ID = vu.CLASS_NAMES, vu.PCU_BY_ID

def open_split(name):
    for cand in ([name] if name != "validation" else ["validation", "val"]):
        try: return load_dataset("iisc-aim/BMD-45", split=cand, streaming=True)
        except Exception as e: err = e
    raise err

# one example crop per class ID -> confirm the class names are in the right order
crops, seen = {}, 0
for ex in open_split("train"):
    img = ex["image"].convert("RGB")
    for (x, y, bw, bh), c in zip(ex["objects"]["bbox"], ex["objects"]["categories"]):
        if c not in crops and bw > 40 and bh > 40: crops[c] = img.crop((int(x), int(y), int(x + bw), int(y + bh)))
    seen += 1
    if len(crops) == len(CLASS_NAMES) or seen >= 600: break
fig, axes = plt.subplots(2, 7, figsize=(18, 5))
for i, ax in enumerate(axes.ravel()):
    ax.axis("off")
    if i in crops: ax.imshow(crops[i]); ax.set_title(f"{i}: {CLASS_NAMES[i]}", fontsize=10)
plt.tight_layout(); savefig("bmd45_classes"); plt.show()

In [ ]:
MAX_SIDE = 1280
SPLITS = [("train", "train", TRAIN_N, 0), ("val", "validation", VAL_N, 0), ("heldout", "validation", HELD_N, VAL_N)]   # (folder, hf split, n, skip)
def export_split(folder, hf_split, n, skip=0):
    img_dir, lbl_dir = YOLO_DIR / "images" / folder, YOLO_DIR / "labels" / folder
    img_dir.mkdir(parents=True, exist_ok=True); lbl_dir.mkdir(parents=True, exist_ok=True)
    ds_ = open_split(hf_split); ds_ = ds_.skip(skip) if skip else ds_
    cc, written = Counter(), 0
    for i, ex in enumerate(ds_):
        if written >= n: break
        img = ex["image"].convert("RGB"); Wd, Ht = img.size; lines = []
        for (x, y, bw, bh), c in zip(ex["objects"]["bbox"], ex["objects"]["categories"]):
            x1, y1, x2, y2 = max(0, x), max(0, y), min(Wd, x + bw), min(Ht, y + bh)
            if x2 - x1 < 2 or y2 - y1 < 2 or not (0 <= c < len(CLASS_NAMES)): continue
            lines.append(f"{int(c)} {(x1 + x2) / 2 / Wd:.6f} {(y1 + y2) / 2 / Ht:.6f} {(x2 - x1) / Wd:.6f} {(y2 - y1) / Ht:.6f}"); cc[int(c)] += 1
        s = MAX_SIDE / max(Wd, Ht)
        if s < 1: img = img.resize((round(Wd * s), round(Ht * s)))
        stem = f"{folder}_{i:06d}"; img.save(img_dir / f"{stem}.jpg", quality=92); (lbl_dir / f"{stem}.txt").write_text("\n".join(lines)); written += 1
    return written, cc
for folder, hf_split, n, skip in SPLITS:
    w_, cc = export_split(folder, hf_split, n, skip); print(f"{folder:8s}: {w_} images | boxes per class:", {CLASS_NAMES[k]: v for k, v in sorted(cc.items())})
(YOLO_DIR / "data.yaml").write_text(f"path: {YOLO_DIR}\ntrain: images/train\nval: images/val\nnames:\n" + "".join(f"  {i}: {n}\n" for i, n in enumerate(CLASS_NAMES)))

In [ ]:
from ultralytics import YOLO
yolo = YOLO("yolo11n.pt")          # yolo11s.pt / yolo11m.pt for a stronger model if you have time
yolo.train(data=str(YOLO_DIR / "data.yaml"), epochs=YOLO_EPOCHS, imgsz=YOLO_IMGSZ, batch=YOLO_BATCH, device=YOLO_DEVICE, seed=SEED,
           patience=10, project=f"{ROOT}/runs", name="bmd45_yolo", exist_ok=True)
BEST = f"{ROOT}/runs/bmd45_yolo/weights/best.pt"
yolo = YOLO(BEST)

### Detector evaluation
mAP on the validation images (overall and per class) and — more relevant to this project — **how well the PCU load estimated from detections tracks the ground-truth PCU load** on held-out frames the model never saw.

In [ ]:
import vision_utils as vu
CLASS_NAMES, PCU_BY_ID = vu.CLASS_NAMES, vu.PCU_BY_ID
vm = yolo.val(data=str(YOLO_DIR / "data.yaml"), split="val", imgsz=YOLO_IMGSZ, verbose=False)
print(f"validation mAP50 = {vm.box.map50:.3f} | mAP50-95 = {vm.box.map:.3f}")
per_class = pd.Series({yolo.names[int(c)]: float(vm.box.maps[int(c)]) for c in vm.box.ap_class_index}).sort_values()
display(per_class.round(3).to_frame("mAP50-95"))

def gt_pcu(label_file):
    cls = [int(l.split()[0]) for l in Path(label_file).read_text().splitlines() if l.strip()]
    return float(PCU_BY_ID[cls].sum()) if cls else 0.0
PCU_REF = np.sort([gt_pcu(f) for f in (YOLO_DIR / "labels" / "val").glob("*.txt")])

held = sorted((YOLO_DIR / "images" / "heldout").glob("*.jpg"))
rows = []
for p_ in held:
    d_ = vu.detect_frame(yolo, str(p_), PCU_REF, imgsz=YOLO_IMGSZ)
    rows.append(dict(gt_pcu=gt_pcu(YOLO_DIR / "labels" / "heldout" / (p_.stem + ".txt")), pred_pcu=d_["pcu"], pred_index=d_["visual_index"]))
cnt = pd.DataFrame(rows)
cnt["gt_index"] = [vu.visual_index(g, PCU_REF) for g in cnt["gt_pcu"]]
pcu_mae = mean_absolute_error(cnt.gt_pcu, cnt.pred_pcu); pcu_r = cnt.gt_pcu.corr(cnt.pred_pcu); pcu_rho = cnt.gt_pcu.corr(cnt.pred_pcu, method="spearman")
idx_mae = mean_absolute_error(cnt.gt_index, cnt.pred_index)
print(f"Held-out frames ({len(cnt)}): PCU MAE {pcu_mae:.2f} | Pearson r {pcu_r:.3f} | Spearman {pcu_rho:.3f} | visual-index MAE {idx_mae:.1f} points")
fig, ax = plt.subplots(1, 2, figsize=(10, 4.5))
lim = [0, max(cnt.gt_pcu.max(), cnt.pred_pcu.max()) + 1]; ax[0].scatter(cnt.gt_pcu, cnt.pred_pcu, s=12, alpha=.5); ax[0].plot(lim, lim, "r--"); ax[0].set_xlabel("ground-truth PCU"); ax[0].set_ylabel("predicted PCU"); ax[0].set_title("PCU load: detections vs ground truth")
ax[1].scatter(cnt.gt_index, cnt.pred_index, s=12, alpha=.5); ax[1].plot([0, 100], [0, 100], "r--"); ax[1].set_xlabel("ground-truth visual index"); ax[1].set_ylabel("predicted visual index"); ax[1].set_title("Visual congestion index")
plt.tight_layout(); savefig("yolo_pcu"); plt.show()

### Package the detector for the app
Saves the trained weights, the PCU reference distribution (to turn a frame's load into a 0–100 index), a handful of sample CCTV frames and a short sample clip, so the Streamlit CCTV tab works out of the box.

In [ ]:
shutil.copy(BEST, f"{PROJECT_DIR}/yolo_bmd45.pt")
json.dump([float(x) for x in PCU_REF], open(f"{PROJECT_DIR}/data/pcu_reference.json", "w"))
from PIL import Image
sample_dir = Path(PROJECT_DIR) / "data" / "sample_frames"; sample_dir.mkdir(parents=True, exist_ok=True)
pick = held[:: max(1, len(held) // 10)][:10]
sample_imgs = []
for k, p_ in enumerate(pick):
    im = Image.open(p_).convert("RGB"); im.thumbnail((960, 960)); im.save(sample_dir / f"frame_{k:02d}.jpg", quality=88); sample_imgs.append(im)
vu.write_clip(sample_imgs, f"{PROJECT_DIR}/data/sample_cctv.mp4", fps=1.0)
print("saved weights, reference, sample frames:", len(sample_imgs), "and clip:", os.path.getsize(f"{PROJECT_DIR}/data/sample_cctv.mp4") // 1024, "KB")

yolo_summary = dict(map50=float(vm.box.map50), map5095=float(vm.box.map), pcu_mae=float(pcu_mae), pcu_pearson=float(pcu_r), pcu_spearman=float(pcu_rho),
                    index_mae=float(idx_mae), train_images=TRAIN_N, epochs=YOLO_EPOCHS, imgsz=YOLO_IMGSZ, gpu=bool(HAS_GPU), lite=not HAS_GPU,
                    best_class=per_class.index[-1], worst_class=per_class.index[0])
meta["vision"] = yolo_summary
json.dump(meta, open(f"{PROJECT_DIR}/meta.json", "w"), indent=1, default=float)
print(yolo_summary)

### The detector end to end: frame → detections → visual index → fused intelligence report
This uses the *saved* weights through the same `vision_utils` code the app uses, including the video path (sample frames every second from the clip).

In [ ]:
det = vu.load_detector(f"{PROJECT_DIR}/yolo_bmd45.pt")
ref = vu.load_reference(f"{PROJECT_DIR}/data/pcu_reference.json")
demo = vu.detect_frame(det, str(held[0]), ref, imgsz=YOLO_IMGSZ)
plt.figure(figsize=(12, 7)); plt.imshow(demo["annotated"]); plt.axis("off")
plt.title(f"{demo['n_vehicles']} vehicles | {demo['pcu']:.1f} PCU | visual congestion index {demo['visual_index']:.0f}/100 | {demo['counts']}", fontsize=10)
savefig("yolo_demo"); plt.show()

clip = vu.video_frames(f"{PROJECT_DIR}/data/sample_cctv.mp4", every_s=1.0, max_frames=12)
per_frame = [vu.detect_frame(det, fr, ref, imgsz=YOLO_IMGSZ) for _, fr in clip]
plt.figure(figsize=(8, 3.2)); plt.plot([t for t, _ in clip], [f["visual_index"] for f in per_frame], "o-"); plt.ylim(0, 100)
plt.xlabel("seconds into clip"); plt.ylabel("visual congestion index"); plt.title("Sample CCTV clip: congestion index over time"); savefig("cctv_timeline"); plt.show()

vision_clip = vu.aggregate(per_frame)
r = mr.intelligence_report(bundle, meta, ranking.loc[0, "road"], "2024-08-12", None, vision_clip)
print(format_report(r))

## Phase 19 — Dashboard
Five sections matching the project brief: **Traffic Intelligence, Road Safety, Traffic Enforcement, Mobility, Prediction**. Everything is interactive Plotly, shown here and saved as one shareable `dashboard.html`.

In [ ]:
mob = obs.groupby("road")[["public_transport_usage", "pedestrian_and_cyclist_count", "parking_usage", "traffic_signal_compliance"]].mean().round(2).reset_index()
road_stats_x = road_stats.merge(mob[["road", "pedestrian_and_cyclist_count", "parking_usage"]], on="road")
DASH = {}

# 1. Traffic intelligence
f = make_subplots(rows=2, cols=2, subplot_titles=("Mean traffic volume by area", "Mean congestion and saturated-day share by area", "Congestion vs capacity utilisation (equal-width bins)", "Speed vs travel-time index (bubble = volume)"))
a = area_stats.sort_values("volume")
f.add_trace(go.Bar(y=a["area"], x=a["volume"], orientation="h", marker_color="#4C72B0", name="volume"), 1, 1)
a2 = area_stats.sort_values("congestion")
f.add_trace(go.Bar(y=a2["area"], x=a2["congestion"], orientation="h", marker_color="#C44E52", name="congestion"), 1, 2)
bb = pd.cut(obs["road_capacity_utilization"], 12); cu = obs.groupby(bb, observed=True).agg(x=("road_capacity_utilization", "mean"), y=("congestion_level", "mean"), n=("congestion_level", "size")).reset_index(drop=True); cu = cu[cu["n"] >= 20]
f.add_trace(go.Scatter(x=cu["x"], y=cu["y"], mode="lines+markers", name="congestion"), 2, 1)
f.add_trace(go.Scatter(x=road_stats["speed"], y=road_stats["travel_time_index"], mode="markers+text", text=road_stats["road"], textposition="top center", textfont=dict(size=8),
                       marker=dict(size=road_stats["volume"] / road_stats["volume"].max() * 30, color=road_stats["congestion"], colorscale="RdYlGn_r", showscale=False), name="roads"), 2, 2)
f.update_layout(height=700, showlegend=False, title_text="Traffic Intelligence"); DASH["Traffic Intelligence"] = f

# 2. Road safety
f = make_subplots(rows=1, cols=3, subplot_titles=("Crashes and deaths per year", "Fatality rate (killed per 100 crashes)", "Fatal share of crashes (%)"))
f.add_trace(go.Bar(x=cr["Year"], y=cr["Total"], name="Total crashes", marker_color="#4C72B0"), 1, 1)
f.add_trace(go.Scatter(x=cr["Year"], y=cr["Killed"], name="Killed", line=dict(color="#C44E52")), 1, 1)
f.add_trace(go.Scatter(x=cr["Year"], y=cr["killed_per_100_crashes"], mode="lines+markers", line=dict(color="#8172B2")), 1, 2)
f.add_trace(go.Scatter(x=cr["Year"], y=cr["fatal_share_pct"], mode="lines+markers", line=dict(color="#DD8452")), 1, 3)
f.update_layout(height=380, showlegend=False, title_text="Road Safety"); DASH["Road Safety"] = f

# 3. Enforcement
f = make_subplots(rows=2, cols=2, subplot_titles=("Cases booked by act", "Total fines collected (Rs crore)", "Automated share of cases (%)", "Towing cases and charges (Rs lakh)"))
for col, nm in [("M.V.Act Cases", "M.V. Act"), ("K.P.Act Cases", "K.P. Act"), ("Towing Cases", "Towing"), ("K.T.C Act Cases", "K.T.C Act"), ("Automation cases", "Automation")]:
    f.add_trace(go.Bar(x=bt["YEAR"], y=bt[col].fillna(0), name=nm), 1, 1)
f.add_trace(go.Bar(x=bt["YEAR"], y=bt["Total Fine Collected"] / 1e7, marker_color="#DD8452", showlegend=False), 1, 2)
f.add_trace(go.Scatter(x=bt["YEAR"], y=bt["automation_share_pct"], mode="lines+markers", line=dict(color="#55A868"), showlegend=False), 2, 1)
f.add_trace(go.Bar(x=bt["YEAR"], y=bt["Towing Cases"], marker_color="#64B5CD", showlegend=False), 2, 2)
f.add_trace(go.Scatter(x=bt["YEAR"], y=bt["Towing Charges"] / 1e5, mode="lines+markers", line=dict(color="#C44E52"), yaxis="y4", showlegend=False), 2, 2)
f.update_layout(height=700, barmode="stack", title_text="Traffic Enforcement"); DASH["Traffic Enforcement"] = f

# 4. Mobility
ms = mob.sort_values("public_transport_usage")
f = make_subplots(rows=2, cols=2, subplot_titles=("Public-transport usage (%) by road", "Pedestrian and cyclist count by road", "Parking usage (%) by road", "Signal compliance (%) by road"))
for (r, c), col, colr in [((1, 1), "public_transport_usage", "#55A868"), ((1, 2), "pedestrian_and_cyclist_count", "#8172B2"), ((2, 1), "parking_usage", "#DD8452"), ((2, 2), "traffic_signal_compliance", "#4C72B0")]:
    s = mob.sort_values(col); f.add_trace(go.Bar(y=s["road"], x=s[col], orientation="h", marker_color=colr, showlegend=False), r, c)
f.update_layout(height=900, title_text="Mobility"); DASH["Mobility"] = f

# 5. Prediction
rk = ranking.sort_values("expected_congestion")
f = make_subplots(rows=1, cols=2, specs=[[{"type": "bar"}, {"type": "bar"}]], subplot_titles=("Expected congestion tomorrow (typical day, all roads)", "What drives estimated congestion (mean |SHAP|, what-if model)"))
f.add_trace(go.Bar(y=rk["road"], x=rk["expected_congestion"], orientation="h", marker_color="#C44E52", showlegend=False), 1, 1)
sb = shap_B.head(10)[::-1]; f.add_trace(go.Bar(y=sb.index, x=sb.values, orientation="h", marker_color="#4C72B0", showlegend=False), 1, 2)
f.update_layout(height=520, title_text="Prediction"); DASH["Prediction"] = f

for name, fig in DASH.items():
    fig.show()

# standalone HTML
kpis = [("Roads monitored", f"{obs['road'].nunique()}"), ("Road-days analysed", f"{len(obs):,}"), ("Mean congestion", f"{obs['congestion_level'].mean():.0f}/100"),
        ("Days at ceiling", f"{obs['saturated'].mean():.0%}"), (f"Killed ({int(cr['Year'].iloc[-1])})", f"{int(cr['Killed'].iloc[-1]):,}"),
        (f"Cases booked ({int(bt['YEAR'].iloc[-1])})", f"{bt['Total Cases Booked'].iloc[-1] / 1e6:.2f}M"), ("What-if model R²", f"{mB['R2']:.2f}"), ("Forecast model R²", f"{mA['R2']:.2f}")]
cards = "".join(f"<div class='k'><div class='v'>{v}</div><div class='l'>{k}</div></div>" for k, v in kpis)
body = "".join(f"<h2>{n}</h2>" + fig.to_html(full_html=False, include_plotlyjs="cdn" if i == 0 else False) for i, (n, fig) in enumerate(DASH.items()))
html = f"""<html><head><meta charset='utf-8'><title>Bengaluru Smart Mobility Intelligence</title><style>
body{{font-family:Segoe UI,Arial;margin:0;background:#f5f7fa;color:#1f2d3d}}header{{background:#14213d;color:#fff;padding:22px 40px}}
.wrap{{padding:20px 40px}}.k{{display:inline-block;background:#fff;border-radius:10px;padding:14px 22px;margin:6px;box-shadow:0 1px 4px #0002;min-width:140px}}
.v{{font-size:26px;font-weight:700;color:#14213d}}.l{{font-size:12px;color:#667}}h2{{margin-top:34px;border-left:5px solid #fca311;padding-left:10px}}</style></head>
<body><header><h1 style='margin:0'>Bengaluru Smart Mobility Intelligence</h1><div>Traffic · Road safety · Enforcement · Mobility · Prediction</div></header>
<div class='wrap'>{cards}{body}</div></body></html>"""
Path(f"{PROJECT_DIR}/dashboard.html").write_text(html, encoding="utf-8")
print("saved dashboard.html")

## Phase 20 — Streamlit deployment (prediction, dashboards and live CCTV detection)
The app is a separate, self-contained package: `app.py`, `model.pkl`, `scaler.pkl`, `mobility_features.py`, `meta.json`, `requirements.txt`, `data/`, `README.md`. First we export the small cleaned data files the app needs.

In [ ]:
obs_export = obs[["date", "area", "road"] + mf.OPS + ["roadwork", "weather_conditions", "saturated", "cong_vs_road"]].copy()
obs_export.to_csv(f"{PROJECT_DIR}/data/traffic_clean.csv", index=False)
cr.to_csv(f"{PROJECT_DIR}/data/crashes_clean.csv", index=False)
bt.to_csv(f"{PROJECT_DIR}/data/btp_clean.csv", index=False)
road_stats_x.merge(mob[["road", "traffic_signal_compliance"]].rename(columns={"traffic_signal_compliance": "compliance_mean"}), on="road").to_csv(f"{PROJECT_DIR}/data/road_stats.csv", index=False)
area_stats.to_csv(f"{PROJECT_DIR}/data/area_stats.csv", index=False)
hot.to_csv(f"{PROJECT_DIR}/data/hotspots.csv", index=False)
json.dump(meta, open(f"{PROJECT_DIR}/meta.json", "w"), indent=1, default=float)
print(sorted(os.listdir(f"{PROJECT_DIR}/data")))

In [ ]:
%%writefile app.py
import os, sys, json, time, tempfile
import numpy as np, pandas as pd
import joblib
from PIL import Image
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import streamlit as st

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, HERE)
import mobility_features as mf

st.set_page_config(page_title="Bengaluru Smart Mobility Intelligence", page_icon="🚦", layout="wide")
BANDS = [(0, 50, "Light"), (50, 75, "Moderate"), (75, 90, "Heavy"), (90, 101, "Severe")]
def band(x):
    return next(n for lo, hi, n in BANDS if lo <= x < hi)

def show(fig):
    try:
        st.plotly_chart(fig, width="stretch")
    except TypeError:
        st.plotly_chart(fig, use_container_width=True)

def table(df):
    try:
        st.dataframe(df, width="stretch", hide_index=True)
    except TypeError:
        st.dataframe(df, use_container_width=True, hide_index=True)

@st.cache_resource
def load_assets():
    return joblib.load(os.path.join(HERE, "model.pkl")), json.load(open(os.path.join(HERE, "meta.json")))

@st.cache_data
def load_tables():
    rd = lambda f, **k: pd.read_csv(os.path.join(HERE, "data", f), **k)
    return dict(traffic=rd("traffic_clean.csv", parse_dates=["date"]), crash=rd("crashes_clean.csv"), btp=rd("btp_clean.csv"),
                road_stats=rd("road_stats.csv"), area_stats=rd("area_stats.csv"), hotspots=rd("hotspots.csv"))

bundle, meta = load_assets()
T = load_tables()
INPUT_COLS, DEFAULTS, ROAD_AREA, RNG = meta["input_cols"], meta["road_defaults"], meta["road_area"], meta["input_ranges"]

import mobility_report as mr

def estimate(road, date, ov):
    return mr.estimate_now(bundle, meta, road, date, ov)

def forecast(road, date, ov):
    return mr.forecast_tomorrow(bundle, meta, road, date, ov)

M = meta["metrics"]
st.title("🚦 Bengaluru Smart Mobility Intelligence")
st.caption("Real Bengaluru traffic, weather, road-safety and enforcement data → congestion forecasts, what-if estimates and planning insights.")
with st.sidebar:
    st.header("Model performance")
    st.metric("What-if estimator R² (held-out test)", f"{M['whatif_test']['R2']:.3f}")
    st.metric("Tomorrow forecast R² (held-out test)", f"{M['forecast_test']['R2']:.3f}")
    st.metric("Road-average baseline R²", f"{M['per_road_mean_test_r2']:.3f}")
    st.metric("Severe-congestion alert AUC", f"{M['alert_test_auc']:.3f}")
    if meta.get("vision"):
        st.metric("CCTV detector mAP50 (BMD-45)", f"{meta['vision']['map50']:.2f}")
    st.caption("Chronological split: trained on earlier dates, tested on the latest period. Tomorrow's congestion is mostly each road's usual level plus noise, so its ceiling is low by nature of the data.")

tab_pred, tab_cctv, tab_traffic, tab_safety, tab_enf, tab_mob, tab_model = st.tabs(
    ["🔮 Prediction", "📹 CCTV Detection", "📍 Traffic Intelligence", "🛡️ Road Safety", "👮 Enforcement", "🚶 Mobility", "🧠 Model & Explainability"])

with tab_pred:
    left, right = st.columns([1, 1.2])
    with left:
        road = st.selectbox("Road / intersection", meta["roads"])
        st.caption(f"Area: {ROAD_AREA[road]}")
        date = st.date_input("Date", value=pd.Timestamp(T["traffic"]["date"].max()).date())
        d = DEFAULTS[road]
        def num(label, col, step):
            lo, hi = RNG[col]
            return st.slider(label, float(lo), float(hi), float(np.clip(d[col], lo, hi)), step=step, key=f"{col}_{road}")
        ov = {"traffic_volume": num("Traffic volume (vehicles)", "traffic_volume", 100.0),
              "incident_reports": num("Incident reports", "incident_reports", 1.0),
              "public_transport_usage": num("Public-transport usage (%)", "public_transport_usage", 1.0),
              "traffic_signal_compliance": num("Signal compliance (%)", "traffic_signal_compliance", 1.0),
              "parking_usage": num("Parking usage (%)", "parking_usage", 1.0),
              "pedestrian_and_cyclist_count": num("Pedestrians + cyclists", "pedestrian_and_cyclist_count", 5.0),
              "rain_sum": num("Rain (mm)", "rain_sum", 0.5), "temp_mean": num("Mean temperature (°C)", "temp_mean", 0.5)}
        ov["roadwork"] = 1.0 if st.checkbox("Roadwork / construction active", value=bool(d["roadwork"]), key=f"rw_{road}") else 0.0
        go_btn = st.button("Predict", type="primary")
    with right:
        if go_btn:
            est = estimate(road, date, ov)
            fc, p_sat = forecast(road, date, ov)
            c1, c2, c3 = st.columns(3)
            c1.metric("What-if estimate (these conditions)", f"{est:.0f}/100", band(est))
            c2.metric("Forecast for tomorrow", f"{fc:.0f}/100", band(fc))
            c3.metric("Chance tomorrow saturates", f"{p_sat:.0%}")
            st.progress(min(max(p_sat, 0.0), 1.0))
            typical = float(DEFAULTS[road]["cong_roll7"])
            st.info(f"**Interpretation.** Under these conditions {road} is estimated at **{est:.0f}/100 ({band(est)})**. "
                    f"Its recent 7-day average is {typical:.0f}. Tomorrow's forecast is **{fc:.0f}/100** with a **{p_sat:.0%}** chance of hitting the ceiling.")
            base = est
            trials = {"Roadwork active": {"roadwork": 1.0}, "Roadwork off": {"roadwork": 0.0},
                      "+3 incidents": {"incident_reports": ov["incident_reports"] + 3}, "Traffic volume +20%": {"traffic_volume": ov["traffic_volume"] * 1.2},
                      "Traffic volume -20%": {"traffic_volume": ov["traffic_volume"] * 0.8}, "Heavy rain (30 mm)": {"rain_sum": 30.0},
                      "Public transport +20%": {"public_transport_usage": ov["public_transport_usage"] * 1.2}}
            sens = pd.Series({k: estimate(road, date, {**ov, **v}) - base for k, v in trials.items()}).sort_values()
            fig = go.Figure(go.Bar(y=sens.index, x=sens.values, orientation="h", marker_color=["#C44E52" if v > 0 else "#4C72B0" for v in sens.values]))
            fig.update_layout(title="What-if: change in estimated congestion (points)", height=330, margin=dict(l=10, r=10, t=40, b=10))
            show(fig)
            if base >= 99:
                st.caption("The estimate is already at the ceiling (100), so adding load cannot raise it further; try lowering the traffic volume to see how the effects behave.")
        else:
            st.write("Set the conditions on the left and press **Predict**.")
        with st.expander("Which features matter most, and why?"):
            for line in meta["explanations"]["whatif"][:6]:
                st.write("•", line)

with tab_cctv:
    st.subheader("📹 CCTV vehicle detection → visual congestion index")
    st.caption("YOLO (trained on BMD-45, 14 Bengaluru vehicle classes) detects vehicles; each is weighted by its passenger-car-unit factor and the frame's load becomes a 0-100 congestion index that is fused into the road's intelligence report.")
    WEIGHTS, REFP = os.path.join(HERE, "yolo_bmd45.pt"), os.path.join(HERE, "data", "pcu_reference.json")
    vision_ok, det, ref = False, None, None
    if not (os.path.exists(WEIGHTS) and os.path.exists(REFP)):
        st.warning("Detector files not found (yolo_bmd45.pt and data/pcu_reference.json). Run Phase 18 of the notebook to create them.")
    else:
        try:
            import vision_utils as vu
            @st.cache_resource
            def load_detector():
                return vu.load_detector(WEIGHTS), vu.load_reference(REFP)
            det, ref = load_detector()
            vision_ok = True
        except Exception as e:
            st.error(f"Could not load the detector ({e}). Install the vision requirements (ultralytics, opencv-python-headless, torch).")
    if vision_ok:
        VM = meta.get("vision", {})
        if VM:
            st.caption(f"Detector quality on held-out BMD-45 frames: mAP50 {VM['map50']:.2f} · PCU-load error {VM['pcu_mae']:.1f} PCU · correlation with true load {VM['pcu_pearson']:.2f}" + (" · LITE (CPU-trained) model" if VM.get("lite") else ""))
        c1, c2, c3 = st.columns([1.3, 1, 1])
        source = c1.radio("Camera source", ["Sample CCTV clip", "Sample frames", "Upload image", "Upload video", "Live stream URL"])
        road_c = c2.selectbox("Pair this camera with a road", meta["roads"], key="cctv_road")
        conf = c3.slider("Detection confidence", 0.10, 0.80, 0.25, 0.05)
        sample_files = sorted(os.listdir(os.path.join(HERE, "data", "sample_frames"))) if os.path.isdir(os.path.join(HERE, "data", "sample_frames")) else []
        sample_choice = up_img = up_vid = None
        every_s, stream_url, auto = 2.0, "", False
        if source == "Sample frames" and sample_files:
            sample_choice = st.selectbox("Frame", sample_files)
        elif source == "Upload image":
            up_img = st.file_uploader("CCTV frame", type=["jpg", "jpeg", "png"])
        elif source == "Upload video":
            up_vid = st.file_uploader("CCTV clip", type=["mp4", "avi", "mov", "mkv"])
            every_s = st.slider("Analyse one frame every (seconds)", 0.5, 10.0, 2.0, 0.5)
        elif source == "Live stream URL":
            stream_url = st.text_input("Stream URL (rtsp://..., http://... MJPEG/HLS, or a webcam index like 0)", key="stream_url")
            auto = st.checkbox("Auto-refresh every 5 seconds")
        trigger = st.button("Capture and analyse" if source == "Live stream URL" else "Analyse", type="primary")
        if source == "Live stream URL" and auto and stream_url:
            trigger = True

        def collect():
            if source == "Sample CCTV clip":
                return [(f"{t:.0f}s", fr) for t, fr in vu.video_frames(os.path.join(HERE, "data", "sample_cctv.mp4"), every_s=1.0, max_frames=12)]
            if source == "Sample frames":
                return [(sample_choice, np.asarray(Image.open(os.path.join(HERE, "data", "sample_frames", sample_choice)).convert("RGB")))] if sample_choice else []
            if source == "Upload image":
                return [(up_img.name, np.asarray(Image.open(up_img).convert("RGB")))] if up_img else []
            if source == "Upload video":
                if not up_vid:
                    return []
                with tempfile.NamedTemporaryFile(suffix=os.path.splitext(up_vid.name)[1], delete=False) as tf:
                    tf.write(up_vid.read())
                return [(f"{t:.0f}s", fr) for t, fr in vu.video_frames(tf.name, every_s=every_s, max_frames=24)]
            if source == "Live stream URL" and stream_url:
                return [(time.strftime("%H:%M:%S"), vu.grab_stream_frame(stream_url))]
            return []

        if trigger:
            try:
                frames = collect()
                if not frames:
                    st.info("Choose or upload something to analyse first.")
                else:
                    prog, results = st.progress(0.0), []
                    for k, (lab, fr) in enumerate(frames):
                        results.append(vu.detect_frame(det, fr, ref, conf=conf))
                        prog.progress((k + 1) / len(frames))
                    st.session_state["cctv"] = dict(labels=[l for l, _ in frames], results=results, road=road_c)
            except Exception as e:
                st.error(f"Could not read the source: {e}")
        S = st.session_state.get("cctv")
        if S:
            res = S["results"]
            vision = vu.aggregate(res)
            ref_date = pd.Timestamp(T["traffic"]["date"].max())
            rep = mr.intelligence_report(bundle, meta, S["road"], ref_date, None, vision)
            k1, k2, k3, k4 = st.columns(4)
            k1.metric("Vehicles per frame", f"{vision['n_vehicles']:.0f}")
            k2.metric("PCU load per frame", f"{vision['pcu']:.1f}")
            k3.metric("Visual congestion index", f"{vision['visual_index']:.0f}/100", mr.band(vision["visual_index"]))
            k4.metric(f"Enforcement priority — {S['road']}", f"{rep['priority_score']:.0f}/100", rep["priority_tier"])
            idx = 0
            if len(res) > 1:
                idx = st.slider("Frame", 0, len(res) - 1, int(np.argmax([r_["visual_index"] for r_ in res])))
                tl = pd.DataFrame({"time": S["labels"], "visual congestion index": [r_["visual_index"] for r_ in res], "PCU": [r_["pcu"] for r_ in res]})
                fig = px.line(tl, x="time", y=["visual congestion index", "PCU"], markers=True, title="Congestion signal across the clip")
                show(fig)
            cc1, cc2 = st.columns([1.6, 1])
            cc1.image(res[idx]["annotated"], caption=f"Frame {S['labels'][idx]} — {res[idx]['n_vehicles']} vehicles, {res[idx]['pcu']:.1f} PCU", channels="RGB")
            counts = pd.Series(res[idx]["counts"]).sort_values()
            if len(counts):
                show(px.bar(counts, orientation="h", title="Vehicles by class (this frame)", labels={"value": "count", "index": ""}))
            st.subheader("Fused intelligence report")
            st.info(f"{S['road']} ({rep['area']}): the camera index ({vision['visual_index']:.0f}/100) is blended 30% into the congestion used for the priority score "
                    f"(model forecast {rep['expected_congestion']:.0f}/100 → blended {rep['congestion_used']:.0f}). Expected incidents {rep['expected_incidents']:.2f}.")
            for line in rep["recommendations"]:
                st.write("•", line)
            st.caption("A single camera view describes one approach, not the whole corridor; the PCU index is relative to annotated BMD-45 frames, not an absolute traffic count.")
            if source == "Live stream URL" and auto:
                time.sleep(5)
                st.rerun()

with tab_traffic:
    tr_ = T["traffic"]
    c1, c2 = st.columns(2)
    a = T["area_stats"].sort_values("pressure_index")
    show(px.bar(a, x="pressure_index", y="area", orientation="h", title="Traffic Pressure Index by area (0-100)"))
    dow = tr_.assign(dow=tr_["date"].dt.day_name()).groupby("dow")["congestion_level"].mean().reindex(["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"])
    show(px.bar(dow, title="Mean congestion by weekday", labels={"value": "congestion", "dow": ""}))
    cb = tr_.assign(b=pd.cut(tr_["road_capacity_utilization"], 12)).groupby("b", observed=True).agg(x=("road_capacity_utilization", "mean"), y=("congestion_level", "mean"), n=("congestion_level", "size")).reset_index(drop=True)
    cb = cb[cb["n"] >= 20]
    show(px.line(cb, x="x", y="y", markers=True, title="Congestion vs capacity utilisation", labels={"x": "capacity utilisation", "y": "congestion"}))
    st.subheader("Locations requiring attention")
    table(T["hotspots"][["area", "road", "congestion", "saturated_share", "compliance", "attention_score", "priority"]].round(2))

with tab_safety:
    cr = T["crash"]
    k1, k2, k3 = st.columns(3)
    k1.metric(f"Killed ({int(cr['Year'].iloc[-1])})", f"{int(cr['Killed'].iloc[-1]):,}")
    k2.metric("Crashes", f"{int(cr['Total'].iloc[-1]):,}")
    k3.metric("Killed per 100 crashes", f"{cr['killed_per_100_crashes'].iloc[-1]:.1f}")
    fig = make_subplots(rows=1, cols=2, subplot_titles=("Crashes and deaths", "Fatality rate (killed per 100 crashes)"))
    fig.add_trace(go.Bar(x=cr["Year"], y=cr["Total"], name="Total crashes"), 1, 1)
    fig.add_trace(go.Scatter(x=cr["Year"], y=cr["Killed"], name="Killed"), 1, 1)
    fig.add_trace(go.Scatter(x=cr["Year"], y=cr["killed_per_100_crashes"], name="Fatality rate", mode="lines+markers"), 1, 2)
    show(fig)
    st.caption("Check whether the latest year in the source file is a full year before reading the most recent change as a trend.")

with tab_enf:
    bt = T["btp"]
    k1, k2, k3 = st.columns(3)
    k1.metric(f"Cases booked ({int(bt['YEAR'].iloc[-1])})", f"{bt['Total Cases Booked'].iloc[-1] / 1e6:.2f} M")
    k2.metric("Fines collected (Rs crore)", f"{bt['Total Fine Collected'].iloc[-1] / 1e7:,.1f}")
    k3.metric("Automated share of cases", f"{bt['automation_share_pct'].iloc[-1]:.0f}%")
    fig = go.Figure()
    for col, nm in [("M.V.Act Cases", "M.V. Act"), ("K.P.Act Cases", "K.P. Act"), ("Towing Cases", "Towing"), ("K.T.C Act Cases", "K.T.C Act"), ("Automation cases", "Automation")]:
        fig.add_trace(go.Bar(x=bt["YEAR"], y=bt[col].fillna(0), name=nm))
    fig.update_layout(barmode="stack", title="Cases booked by act")
    show(fig)
    show(px.bar(bt, x="YEAR", y=bt["Total Fine Collected"] / 1e7, title="Total fines collected (Rs crore)", labels={"y": "Rs crore"}))

with tab_mob:
    rs = T["road_stats"]
    for col, ttl in [("public_transport", "Public-transport usage (%) by road"), ("pedestrian_and_cyclist_count", "Pedestrian and cyclist count by road"),
                     ("parking_usage", "Parking usage (%) by road"), ("compliance", "Signal compliance (%) by road")]:
        s = rs.sort_values(col)
        show(px.bar(s, x=col, y="road", orientation="h", title=ttl))

with tab_model:
    st.subheader("Task A — forecast tomorrow's congestion")
    ca = pd.DataFrame(meta["comparison"]["forecast"])[["Model", "Type", "cv_r2", "Test MAE", "Test RMSE", "Test R2"]].sort_values("cv_r2", ascending=False)
    table(ca.round(3))
    st.subheader("Task B — what-if congestion estimator")
    cb_ = pd.DataFrame(meta["comparison"]["whatif"])[["Model", "Type", "cv_r2", "Test MAE", "Test RMSE", "Test R2"]].sort_values("cv_r2", ascending=False)
    table(cb_.round(3))
    imp = pd.Series(meta["importance"]["whatif"]).sort_values()
    show(px.bar(imp, orientation="h", title="What drives estimated congestion (mean |SHAP|)", labels={"value": "mean |SHAP|", "index": ""}))
    st.subheader("What the models learned")
    for line in meta["explanations"]["whatif"]:
        st.write("•", line)
    st.subheader("Key mobility insights")
    for line in meta.get("insights", []):
        st.write("•", line)
    st.warning("Limitations: observational data from 16 roads over about 2.5 years; congestion is capped at 100 so extreme days are indistinguishable; tomorrow's forecast is limited by day-to-day noise; yearly crash/enforcement tables are small (15-19 rows) and correlations are descriptive, not causal.")

In [ ]:
import importlib.metadata as im
txt = " ".join(repr(m) for m in [forecast_prod, alert_prod, whatif_prod])
def _pin(pkg):
    try: return f"{pkg}=={im.version(pkg)}"
    except Exception: return pkg
reqs = ["--extra-index-url https://download.pytorch.org/whl/cpu", "streamlit>=1.35", _pin("pandas"), _pin("numpy"), _pin("scikit-learn"), _pin("joblib"),
        "plotly>=5", "Pillow", _pin("ultralytics"), "opencv-python-headless", "torch", "torchvision"]
for key, pkg in [("XGB", "xgboost"), ("LGBM", "lightgbm"), ("CatBoost", "catboost")]:
    if key in txt: reqs.append(_pin(pkg))
Path(f"{PROJECT_DIR}/requirements.txt").write_text("\n".join(reqs) + "\n")

readme = f"""# Bengaluru Smart Mobility Intelligence

Real Bengaluru traffic, weather, road-safety and traffic-enforcement data turned into congestion forecasts, a what-if congestion estimator and planning insights.

## What it does
* **What-if estimator** (R² {mB['R2']:.2f} on a held-out later period; traffic volume alone reaches {vol_only_r2:.2f}, so it is mostly a volume-to-congestion curve): estimates congestion from volume, incidents, roadwork, public-transport usage, weather and calendar for a chosen road.
* **Tomorrow forecast + alert**: expected congestion and the probability that tomorrow saturates at 100.
* **Mobility intelligence**: pressure by area, time patterns, capacity utilisation, incidents, weather, roadwork, hotspots, road safety, enforcement.
* **CCTV detection tab**: a YOLO detector trained on BMD-45 (mAP50 {yolo_summary['map50']:.2f}) finds vehicles in an uploaded image, a video clip or a live stream URL, converts the load to a PCU-weighted visual congestion index and fuses it into the road's intelligence report.
* **Streamlit app** with prediction, interpretation, charts, CCTV detection and model explainability.

## Folder structure
```
bengaluru_smart_mobility/
├── app.py                  Streamlit application
├── mobility_features.py    shared feature code (training and serving)
├── mobility_report.py      shared report logic (forecast, risk, priority, vision fusion)
├── vision_utils.py         YOLO detection, PCU load, video / stream helpers
├── yolo_bmd45.pt           trained YOLO weights (BMD-45)
├── model.pkl               saved models (what-if estimator, forecast, alert, incident risk)
├── scaler.pkl              scaler for the numeric driver inputs
├── meta.json               metrics, importances, defaults, insights
├── roads.json
├── requirements.txt
├── data/                   cleaned data for the dashboard + original CSVs
├── models/  figures/  docs/  notebooks/
└── dashboard.html          standalone interactive dashboard
```

## Run locally
```
pip install -r requirements.txt
streamlit run app.py
```

## Deploy (free)
The CCTV tab needs PyTorch + OpenCV, so prefer a host with enough memory:
1. **Hugging Face Spaces** (recommended): create a Space with SDK *Streamlit* and upload this folder (the 2 GB-RAM CPU tier is enough for the small YOLO model).
2. **Streamlit Community Cloud**: push this folder to GitHub and set the main file to `app.py` (memory is tighter; the app still works because the detector loads lazily, and the other tabs do not need PyTorch).
3. Live RTSP cameras must be reachable from the host (a public HTTP/MJPEG URL, or run the app on a machine inside the camera network).

## Honest limitations
Tomorrow's congestion on a road is mostly its usual level plus day-to-day noise (road average alone gives R² {rm['Test R2']:.2f}); the high-R² model is a what-if *estimator*, not a forecast. The CCTV index describes one camera view, is relative to annotated BMD-45 frames and is not geo-linked to the traffic table (the user pairs a camera with a road). Data covers 16 roads and about 2.5 years; congestion is capped at 100; the yearly crash and enforcement tables are small and their relationships are descriptive, not causal.
"""
Path(f"{PROJECT_DIR}/README.md").write_text(readme, encoding="utf-8")
print(open(f"{PROJECT_DIR}/requirements.txt").read())

In [ ]:
# smoke-test the app headlessly (same engine Streamlit uses for its own tests)
try:
    from streamlit.testing.v1 import AppTest
    at = AppTest.from_file(f"{PROJECT_DIR}/app.py", default_timeout=180).run()
    assert not at.exception, at.exception
    [b for b in at.button if b.label == "Predict"][0].click().run()
    assert not at.exception, at.exception
    cctv_btn = [b for b in at.button if b.label == "Analyse"]
    if cctv_btn:
        cctv_btn[0].click().run()
        assert not at.exception, at.exception
    print("Streamlit app smoke test passed: loads, Predict works" + (", and the CCTV tab analysed the sample clip." if cctv_btn else " (CCTV tab inactive: detector files or vision libraries missing)."))
    print([m.label for m in at.metric][:10])
except ImportError:
    print("streamlit not installed in this runtime; run  !pip install -q streamlit  then re-run this cell")

## Phase 21 — Documentation, presentation, resume text and final package
Generated from this run's actual results, so every number matches the notebook.

In [ ]:
def fm(m): return f"MAE {m['MAE']:.2f} · RMSE {m['RMSE']:.2f} · R² {m['R2']:.3f}"
report = f"""# Bengaluru Smart Mobility Intelligence — Project Report

## 1. Problem
Use real Bengaluru traffic, weather, road-safety and enforcement data to (A) forecast next-day congestion, (B) estimate congestion under given conditions, and (C) turn the data into planning intelligence.

## 2. Data and roles
| Dataset | Rows | Role |
|---|---|---|
| Traffic (16 roads, {traffic['Date'].min().date()} to {traffic['Date'].max().date()}) | {len(traffic):,} | prediction + insight |
| Weather (hourly) | {len(weather):,} | feature source |
| Road crashes 2007-2025 | {len(crash)} | road-safety insight |
| BTP cases and fines | {len(btp)} | enforcement insight |

## 3. Data-quality and leakage audit
* The earlier "next-day" target mixed horizons: only {100 * (old['next_gap'] == 1).mean():.0f}% of rows were truly the next day, and {100 * (y_old == 100).mean():.0f}% of values sit at the ceiling of 100.
* Interpolated 'today' rows leak the next day: the same model scores far higher with them (see the leakage demonstration) and is not real skill. They are excluded.
* `Environmental Impact` duplicates `Traffic Volume` (r = 1.0) and is dropped.

## 4. Method
Chronological split (development to {dev['target_date'].max().date()}, test from {test['target_date'].min().date()}), expanding-window time-series cross-validation, random-search tuning, one-standard-error simplicity rule for model choice.

## 5. Results
**Task A — tomorrow's congestion:** final model {final_A_name}: {fm(mA)}. The per-road average alone scores R² {rm['Test R2']:.3f}; persistence scores {res_A.loc[res_A['Model'].str.startswith('Persistence'), 'Test R2'].iloc[0]:.3f}. Alert classifier ({alert_name}): test AUC {roc_auc_score(sat_test, p_alert):.3f}.

**Task B — what-if estimator:** final model {final_B_name}: {fm(mB)} (cross-validated R² {pick_B['cv_r2']:.3f}). Reality check: traffic volume alone reaches R² {vol_only_r2:.3f} — in this dataset congestion is almost a saturating function of volume, so the other drivers add little.

## 6. Explainability
{chr(10).join('* ' + l for l in lines_B[:6])}

## 7. Mobility insights
{chr(10).join('* ' + i for i in INSIGHTS)}

## 8. Vision — CCTV vehicle detection
YOLO11n trained on BMD-45 ({yolo_summary['train_images']} training images, {yolo_summary['epochs']} epochs, {yolo_summary['imgsz']}px{', LITE CPU configuration' if yolo_summary['lite'] else ', T4 GPU'}): validation mAP50 {yolo_summary['map50']:.3f}, mAP50-95 {yolo_summary['map5095']:.3f}. On held-out frames the PCU load estimated from detections has MAE {yolo_summary['pcu_mae']:.2f} PCU and correlation {yolo_summary['pcu_pearson']:.2f} with the annotated load; the visual congestion index is within {yolo_summary['index_mae']:.0f} points of the annotation-based index on average. Weakest class: {yolo_summary['worst_class']}; strongest: {yolo_summary['best_class']}. The index is fused into the intelligence report and served in the app's CCTV tab (image, video, live stream).

## 9. Deep-learning comparison
GRU R² {seq_res.set_index('Model').loc['GRU', 'R2']:.3f}, LSTM R² {seq_res.set_index('Model').loc['LSTM', 'R2']:.3f} vs per-road mean {seq_res.set_index('Model').loc['Per-road mean', 'R2']:.3f} on identical leak-free 7-day windows: sequence models do not find a temporal pattern the simple models miss.

## 10. Deployment
Streamlit app (`app.py`) with prediction, interpretation, a CCTV detection tab, five analysis tabs and model explainability; models in `model.pkl`; shared feature code in `mobility_features.py`.

## 11. Limitations
The CCTV index covers one camera view and is relative to annotated frames. Tomorrow's congestion is largely the road's usual level plus noise, which caps forecast R²; the what-if model is an estimator, not a forecast; congestion is capped at 100; small yearly tables; observational data (associations, not causation).
"""
Path(f"{PROJECT_DIR}/docs/PROJECT_REPORT.md").write_text(report, encoding="utf-8")

resume = f"""# Resume bullets
* Built an end-to-end ML system on real Bengaluru traffic, weather, road-safety and enforcement data (8.9k road-days, 16 roads): leak-free feature pipeline, time-series cross-validation, tuned gradient-boosting models and SHAP explanations; what-if congestion estimator reached R² {mB['R2']:.2f} on a held-out later period (and I showed that traffic volume alone explains R² {vol_only_r2:.2f} of it, so the model is honest about being volume-driven).
* Audited and corrected target leakage (mixed forecast horizons and interpolated inputs that inflated R² by about 3x), replacing a random split with a chronological one — and reported the honest forecast ceiling (road average R² {rm['Test R2']:.2f}) instead of chasing metrics.
* Trained a YOLO11 vehicle detector on 14 Bengaluru CCTV vehicle classes (BMD-45; mAP50 {yolo_summary['map50']:.2f}) and converted detections into a PCU-weighted visual congestion index, served in the app via image upload, video and live-stream input and fused into the intelligence report.
* Delivered a severe-congestion alert (AUC {roc_auc_score(sat_test, p_alert):.2f}), mobility intelligence (area pressure, hotspots, incident-risk model, road-safety and enforcement analytics) and a deployed Streamlit app with interactive what-if analysis.

# LinkedIn project description
**Bengaluru Smart Mobility Intelligence** — I combined real Bengaluru traffic, weather, road-crash and traffic-police enforcement data into a decision-support system. Along the way I found that my first model's "good" scores came from data leakage, rebuilt the evaluation chronologically, and learned how much of tomorrow's congestion is genuinely predictable (little — most of it is which road it is). The result: a what-if congestion estimator (R² {mB['R2']:.2f}), a severe-congestion alert, hotspot and risk analytics, SHAP explanations and a Streamlit app. Computer vision: a YOLO detector on Bengaluru CCTV frames turns live camera views into a congestion signal inside the same app. Tech: Python, scikit-learn, XGBoost/LightGBM/CatBoost, SHAP, PyTorch/Ultralytics YOLO, OpenCV, Plotly, Streamlit.

# Interview talking points
* **Why is forecast R² low?** Tomorrow's congestion is mostly the road's usual level plus noise; persistence is worse than predicting the average, and calendar and weather add nothing detectable. I verified that rather than adding leaky features.
* **What leakage did you find?** The old target mixed 1-17 day horizons, and interpolated "today" rows carried tomorrow's value; fixing both cut R² from the inflated range to the real one.
* **Why is the what-if R² high and is that legitimate?** Evaluated on a later period it generalises, but a volume-only model already reaches R² {vol_only_r2:.2f}: congestion in this data is almost a saturating function of traffic volume. It is a planning estimator, not a forecast, and not a rich multi-factor model.
* **Why chronological split, why 1-SE rule?** Realistic evaluation and the simplest model that is statistically as good.
* **Why SHAP?** Direction and magnitude of each driver, validated against permutation importance.
"""
Path(f"{PROJECT_DIR}/docs/RESUME_LINKEDIN.md").write_text(resume, encoding="utf-8")
print(resume[:900])

In [ ]:
from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.dml.color import RGBColor

prs = Presentation(); prs.slide_width, prs.slide_height = Inches(13.333), Inches(7.5)
NAVY, GOLD = RGBColor(0x14, 0x21, 0x3D), RGBColor(0xFC, 0xA3, 0x11)
def slide(title, bullets=None, image=None, note=None):
    s = prs.slides.add_slide(prs.slide_layouts[6])
    bar = s.shapes.add_shape(1, 0, 0, prs.slide_width, Inches(1.1)); bar.fill.solid(); bar.fill.fore_color.rgb = NAVY; bar.line.fill.background()
    tb = s.shapes.add_textbox(Inches(0.5), Inches(0.2), Inches(12.3), Inches(0.8)); tb.text_frame.text = title
    r = tb.text_frame.paragraphs[0].runs[0]; r.font.size = Pt(30); r.font.bold = True; r.font.color.rgb = RGBColor(255, 255, 255)
    width = Inches(6.2) if image else Inches(12.3)
    if bullets:
        box = s.shapes.add_textbox(Inches(0.6), Inches(1.5), width, Inches(5.5)); tf = box.text_frame; tf.word_wrap = True
        for i, b in enumerate(bullets):
            p = tf.paragraphs[0] if i == 0 else tf.add_paragraph(); p.text = "• " + b; p.font.size = Pt(18); p.space_after = Pt(10)
    if image and os.path.exists(image):
        s.shapes.add_picture(image, Inches(7.0), Inches(1.5), width=Inches(5.9))
    if note:
        n = s.shapes.add_textbox(Inches(0.6), Inches(6.9), Inches(12), Inches(0.4)); n.text_frame.text = note; n.text_frame.paragraphs[0].font.size = Pt(11)
    return s
F = lambda n: f"{PROJECT_DIR}/figures/{n}.png"
slide("Bengaluru Smart Mobility Intelligence", ["Real Bengaluru traffic, weather, road-safety and enforcement data", "Forecasts, what-if estimates, hotspot and risk analytics, Streamlit app", "AI/ML capstone"])
slide("Problem and data", ["Two tasks: forecast tomorrow's congestion; estimate congestion under given conditions", f"Traffic: {len(traffic):,} road-days, 16 roads, {traffic['Date'].min().date()} to {traffic['Date'].max().date()}", "Weather = features; crashes and BTP enforcement = insight datasets", "Vision module (BMD-45 + YOLO) as an optional extension"])
slide("A data-leakage audit changed the results", [f"Old target: only {100 * (old['next_gap'] == 1).mean():.0f}% truly next-day; {100 * (y_old == 100).mean():.0f}% at the ceiling of 100", "Interpolated 'today' rows leaked tomorrow's value (R² looked ~3x better)", "Random split replaced by chronological split", "Environmental Impact = duplicate of Traffic Volume"], F("target_distribution"))
slide("Model results", [f"What-if estimator ({final_B_name}): R² {mB['R2']:.3f}, MAE {mB['MAE']:.1f} (volume alone: {vol_only_r2:.3f})", f"Tomorrow forecast ({final_A_name}): R² {mA['R2']:.3f} vs road-average {rm['Test R2']:.3f}", f"Severe-congestion alert: AUC {roc_auc_score(sat_test, p_alert):.2f}", "8 algorithms compared, tuned, time-series cross-validated; simplest model within 1 SE chosen"], F("model_comparison_B"))
slide("Explainable AI", lines_B[:4], F("xai_B"))
slide("Mobility intelligence", INSIGHTS[:4], F("pressure_index"))
slide("CCTV vehicle detection (YOLO + BMD-45)", [f"YOLO11n, 14 vehicle classes; mAP50 {yolo_summary['map50']:.2f}, mAP50-95 {yolo_summary['map5095']:.2f}", f"PCU load from detections tracks the annotated load (r = {yolo_summary['pcu_pearson']:.2f}, MAE {yolo_summary['pcu_mae']:.1f} PCU)", "PCU-weighted visual congestion index (0-100) per frame", "Streamlit CCTV tab: image upload, video clip, live stream URL; fused into the priority report"], F("yolo_demo"))
slide("Hotspots and risk", [f"Highest attention: {', '.join(hot['road'].head(3))}", INSIGHTS[5], INSIGHTS[9], INSIGHTS[10]], F("hotspots"))
slide("Dashboard and deployment", ["Interactive Plotly dashboard (5 sections) + standalone HTML", "Streamlit app: prediction, CCTV detection tab, analysis tabs, explainability", "Shared feature code prevents train/serve skew", "Deploy: Streamlit Community Cloud or Hugging Face Spaces"])
slide("Limitations and next steps", ["Tomorrow's congestion is mostly the road's usual level plus noise", "Congestion capped at 100; 16 roads; small yearly tables; associations not causation", "Next: more roads/years, live weather API, camera-based congestion index, per-road models"])
prs.save(f"{PROJECT_DIR}/docs/Bengaluru_Smart_Mobility_Intelligence.pptx")
print("saved docs:", sorted(os.listdir(f"{PROJECT_DIR}/docs")))

## Final scoreboard and project package

In [ ]:
print("=" * 82)
print(f"TASK B  what-if congestion estimator : {final_B_name:26s} test {fm(mB)}")
print(f"TASK A  tomorrow's congestion        : {final_A_name:26s} test {fm(mA)}")
print(f"        (per-road mean baseline R2 {rm['Test R2']:.3f} | persistence R2 {res_A.loc[res_A['Model'].str.startswith('Persistence'), 'Test R2'].iloc[0]:.3f} | GRU {seq_res.set_index('Model').loc['GRU', 'R2']:.3f} | LSTM {seq_res.set_index('Model').loc['LSTM', 'R2']:.3f})")
print(f"ALERT   P(saturated tomorrow)        : {alert_name:26s} test AUC {roc_auc_score(sat_test, p_alert):.3f}")
print(f"RISK    incident Poisson model       : deviance {mean_poisson_deviance(yt, yp):.3f} vs baseline {mean_poisson_deviance(yt, base_mean):.3f}")
print(f"VISION  YOLO CCTV detector           : mAP50 {yolo_summary['map50']:.3f} | PCU MAE {yolo_summary['pcu_mae']:.2f} | r {yolo_summary['pcu_pearson']:.2f}" + ("  [LITE: CPU-trained]" if yolo_summary['lite'] else ""))
print("=" * 82)
print("Project folder:", PROJECT_DIR); print(sorted(os.listdir(PROJECT_DIR)))

In [ ]:
# ---- launch the app from Colab and open a public link (uncomment the 4 lines) ----
# !pip install -q streamlit
# !streamlit run /content/bengaluru_smart_mobility/app.py --server.port 8501 --server.headless true &>/content/streamlit.log &
# !curl -s ipv4.icanhazip.com        # this is the 'tunnel password' the next page asks for
# !npx --yes localtunnel --port 8501 # open the https://....loca.lt link it prints
# ---- DOWNLOAD the whole project as a zip ----
import shutil
zip_path = shutil.make_archive("/content/bengaluru_smart_mobility_project", "zip", PROJECT_DIR)
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print("zip created at", zip_path)